# Public Task 1 reproducibility notebook

This is a public generated reproducibility copy, not an original downloaded execution record. Code cells are unexecuted; actual observations come from verified embedded logs. Live paid calls are disabled by default. Private identifiers and metadata are omitted.

# 作业2 Task 1：完整实验与可复现结果

运行路线：**固定数据与代码 → 独立开发检查 → 三种提示与Agent正式对比 → 离线分析 → 下载结果**。

本notebook包含全部执行源码和此次正式数据，不依赖临时下载链接。模型通过DeepSeek官方API远程运行，CPU运行时足够。

为了避免打开交付本时重复付费，`RUN_LIVE_EXPERIMENTS`与`RUN_SUPPLEMENTAL_LIVE`默认均为`False`。已附日志会按SHA校验还原，默认运行全部即可离线复算主实验与补充研究、展示和导出，不会请求Key或调用模型。

只有需要开始/恢复尚未完成的实验时才改成`True`，并在隐藏输入框填写自己的Key。同配置已完成项会跳过。独立重跑必须同时设置新的`PROJECT_DIR`、`RESTORE_EXISTING_RESULTS=False`和`RUN_LIVE_EXPERIMENTS=True`，保留旧结果且不能按答错情况选择重跑。Key不会写进源文件、日志或ZIP。

**交付本是在本地构建的，代码格没有执行记录。**后面的实际成绩（若已附）来自已下载日志的只读汇总。不要把未运行的代码格或开发题成绩当作正式实验结果。

## 实际运行摘要（只读导入）

下表由已下载日志生成；它不是本notebook代码格的执行输出。日志、指纹及源文件哈希保存在配套源码ZIP中。所有代码格仍保持未执行状态。

| 阶段 | 方法 | 终结结果数/有效输入数 | 正确数 | 主集合准确率 | 敏感性准确率 | API尝试次数 | 已知token |
|---|---|---:|---:|---:|---:|---:|---:|
| development | zero_shot | 3/3 | 3 | 100.00% | 100.00% | 3 | 619 |
| development | few_shot | 3/3 | 2 | 66.67% | 66.67% | 3 | 1262 |
| development | cot | 3/3 | 3 | 100.00% | 100.00% | 3 | 807 |
| development | agent | 3/3 | 3 | 100.00% | 100.00% | 9 | 3063 |
| formal | zero_shot | 97/97 | 79 | 81.44% | 80.00% | 97 | 24807 |
| formal | few_shot | 97/97 | 80 | 82.47% | 81.11% | 97 | 45662 |
| formal | cot | 97/97 | 78 | 80.41% | 78.89% | 97 | 33158 |
| formal | agent | 97/97 | 82 | 84.54% | 84.44% | 301 | 150410 |

开发题仅用于流程/格式验证，不估计药师考试能力。正式主集合为97条有文本选项的记录，预注册敏感性集合为90条；原始100条全部保留。API/解析失败计错，未完成项不会被写成最终准确率。

已知token不等于账单。若存在未知用量，需要查阅日志并在成本说明中保留该限制。

### 主实验之后追加的研究（Post-hoc）

A为16道独立人工封闭世界协议检查（8道设计题、8道固定probe），B为95个原始合法初答状态上的单步反馈消融。它们不替代主实验，也不代表完整Agent消融。

| 研究 | 条件 | 集合 | 正确数/分母 | 准确率 | 格式合法率 | API次数 |
|---|---|---|---:|---:|---:|---:|
| A | cot_v1 | all_16 | 13/16 | 81.25% | 100.00% | 16 |
| A | cot_v1 | held_out_probe | 8/8 | 100.00% | 100.00% | 8 |
| A | cot_v2 | all_16 | 16/16 | 100.00% | 100.00% | 16 |
| A | cot_v2 | held_out_probe | 8/8 | 100.00% | 100.00% | 8 |
| B | feedback | conditional_95 | 78/95 | 82.11% | 97.89% | 95 |
| B | feedback | overall_97 | 78/97 | 80.41% | 95.88% | 95 |
| B | null_feedback | conditional_95 | 78/95 | 82.11% | 98.95% | 95 |
| B | null_feedback | overall_97 | 78/97 | 80.41% | 96.91% | 95 |

两项新增预指定比较为A全部16题与B条件95题，使用双侧精确McNemar检验并在这两个比较内做Holm校正；probe8与overall97只是重叠的描述子集。
补充实际调用：222；已知token：126527；未知用量调用：1。
- A_all_16: McNemar p=0.25；Holm2 p=0.5。
- B_conditional_95: McNemar p=1；Holm2 p=1。

## 1. 安装运行依赖

从标准PyPI安装OpenAI兼容SDK和Matplotlib。代码连接的是DeepSeek官方服务，不需要OpenAI Key。此格没有模型调用。

In [ ]:
%pip -q install --index-url https://pypi.org/simple openai matplotlib

## 2. 写入冻结源码与数据

代码按原始字节还原并验证SHA-256。真实日志采用压缩包内嵌，验证压缩包及逐文件SHA后，只还原白名单路径。源码和原始日志已有不同内容时停止，避免覆盖旧实验。派生分析若已存在则保留，交给第7步重算，所以可以重复运行。100条原始数据完整保留，模型请求只取题干、题型和非空选项。

In [ ]:
import base64
import hashlib
import io
import json
import os
import sys
import zipfile
from pathlib import Path
from importlib.metadata import version
PROJECT_DIR = Path('/content/task1_assignment_repro')
RESTORE_EXISTING_RESULTS = True
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_FILES = {'runner.py': {'base64': '...', 'sha256': '4c763ac4808f06d54f4d63b9f1ef367fa4eb2f581541ebc9aa2f708ca80aef2c'}, 'cloud_driver.py': {'base64': 'IiIiQ29sYWIg5a6e6aqM5YWl5Y+j77yb5a+85YWl5Y+q6K+75pWw5o2u77yMQVBJIOiwg+eUqOeUseS4pOS4quaYjuehrueahOWHveaVsOinpuWPkeOAgiIiIgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHVybGxpYi5yZXF1ZXN0Cgpmcm9tIHJ1bm5lciBpbXBvcnQgRXhwZXJpbWVudENvbmZpZywgRXhwZXJpbWVudFJ1bm5lciwgTUVUSE9EUwoKREFUQV9VUkwgPSAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL05MUC1Db3Vyc2UtQ1VIS1NaL05MUC1jb3Vyc2UtY3Voa3N6LmdpdGh1Yi5pby9tYWluL0Fzc2lnbm1lbnRzL0Fzc2lnbm1lbnQyL3Rhc2sxL2RhdGEvMS5leGFtLmpzb24iClJPT1QgPSBQYXRoKCJ0YXNrMV9hc3NpZ25tZW50IikKUk9PVC5ta2RpcihleGlzdF9vaz1UcnVlKQpEQVRBX1BBVEggPSBQYXRoKCJ0YXNrMV9zbW9rZS8xLmV4YW0uanNvbiIpCmlmIG5vdCBEQVRBX1BBVEguZXhpc3RzKCk6CiAgICBEQVRBX1BBVEggPSBST09UIC8gIjEuZXhhbS5qc29uIgogICAgaWYgbm90IERBVEFfUEFUSC5leGlzdHMoKToKICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oREFUQV9VUkwsIHRpbWVvdXQ9MzApIGFzIHJlc3BvbnNlOgogICAgICAgICAgICBEQVRBX1BBVEgud3JpdGVfYnl0ZXMocmVzcG9uc2UucmVhZCgpKQpEQVRBID0ganNvbi5sb2FkcyhEQVRBX1BBVEgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQphc3NlcnQgbGVuKERBVEEpID09IDEwMCwgIuato+W8j+aVsOaNruS4jeaYrzEwMOadoe+8jOivt+WFiOajgOafpeOAgiIKKFJPT1QgLyAiMS5leGFtLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoREFUQSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCgojIOeLrOeri+S6uuW3peaehOmAoOeahOW8gOWPkemimO+8jOS7heeUqOS6juajgOafpei0n+WQkemXruazleOAgeWNleWkmumAieWSjOagvOW8j++8m+S4jeS8sOiuoeecn+WunuiAg+ivleaAp+iDveOAggojIOWug+S7rOS4jeaYryBmZXctc2hvdCDmvJTnpLrpopjvvIzkuZ/kuI3ku47mraPlvI8xMDDpopjlj4rlhbbmoIflh4bop6PmnpDnlJ/miJDjgIIKREVWX0RBVEEgPSBbCiAgICB7InF1ZXN0aW9uX3R5cGUiOiAi5pyA5L2z6YCJ5oup6aKYIiwgInF1ZXN0aW9uIjogIuS7heS9nOWNleS9jeaNoueul+ajgOafpe+8muS7peS4i+WTqumhueS4jeetieS6jjLlhYvvvJ8iLAogICAgICJvcHRpb24iOiB7IkEiOiAiMjAwMOavq+WFiyIsICJCIjogIjAuMDAy5Y2D5YWLIiwgIkMiOiAiMjAwMDAwMOW+ruWFiyIsICJEIjogIjIwMOavq+WFiyJ9LAogICAgICJhbnN3ZXIiOiAiRCIsICJhbmFseXNpcyI6ICLku4XkvpvlvIDlj5Hor4TliIbvvJoyMDDmr6vlhYvnrYnkuo4wLjLlhYvjgIIiLCAic291cmNlIjogIueLrOeri+S6uuW3peaehOmAoOW8gOWPkemimGRldjAxIn0sCiAgICB7InF1ZXN0aW9uX3R5cGUiOiAi5aSa6aG56YCJ5oup6aKYIiwgInF1ZXN0aW9uIjogIuS7heS9nOS9k+enr+aNoueul+ajgOafpe+8muS4i+WIl+WTquS6m+S9k+enr+etieS6jjPljYfvvJ/or7fpgInmi6nlhajpg6jmraPnoa7pgInpobnjgIIiLAogICAgICJvcHRpb24iOiB7IkEiOiAiMzAwMOavq+WNhyIsICJCIjogIjMwMOavq+WNhyIsICJDIjogIjAuMDAz56uL5pa557GzIiwgIkQiOiAiMzAwMDAwMOW+ruWNhyIsICJFIjogIjAuM+WNhyJ9LAogICAgICJhbnN3ZXIiOiAiQUNEIiwgImFuYWx5c2lzIjogIuS7heS+m+W8gOWPkeivhOWIhu+8mjHljYc9MTAwMOavq+WNhz0wLjAwMeeri+aWueexsz0xMDAwMDAw5b6u5Y2H44CCIiwgInNvdXJjZSI6ICLni6znq4vkurrlt6XmnoTpgKDlvIDlj5HpophkZXYwMiJ9LAogICAgeyJxdWVzdGlvbl90eXBlIjogIumFjeS8jemAieaLqemimCIsICJxdWVzdGlvbiI6ICLku4XkvZzlupPlrZjnrpfmnK/mo4Dmn6XvvJrljp/mnIkxODDku7bnianlk4HvvIzmlLbliLAzNeS7tu+8jOWPkeWHujYw5Lu277yM5rKh5pyJ5YW25LuW5Ye65YWl5bqT44CC5Ymp5L2Z5pWw6YeP5a+55bqU5ZOq5Liq6YCJ6aG577yfIiwKICAgICAib3B0aW9uIjogeyJBIjogIjIxNeS7tiIsICJCIjogIjE1NeS7tiIsICJDIjogIjE0NeS7tiIsICJEIjogIjI3NeS7tiJ9LAogICAgICJhbnN3ZXIiOiAiQiIsICJhbmFseXNpcyI6ICLku4XkvpvlvIDlj5Hor4TliIbvvJoxODArMzUtNjA9MTU144CCIiwgInNvdXJjZSI6ICLni6znq4vkurrlt6XmnoTpgKDlvIDlj5HpophkZXYwMyJ9LApdCgoKZGVmIGNvbXBhY3Rfc3VtbWFyeShzdW1tYXJ5KToKICAgICIiIuS4jeWxleekuuWHreaNruaIluWuouaIt+err++8jOWPquWxleekuuWQhOaWueazleeahOW3suWujOaIkOeKtuaAgeOAgiIiIgogICAgZm9yIG1ldGhvZCwgdmFsdWVzIGluIHN1bW1hcnlbIm1ldGhvZHMiXS5pdGVtcygpOgogICAgICAgIHByaW50KG1ldGhvZCwgIueKtuaAgT0iLCB2YWx1ZXNbInN0YXR1c19jb3VudHMiXSwKICAgICAgICAgICAgICAi5q2j56Gu5pWwPSIsIHZhbHVlc1siY29ycmVjdF9jb3VudCJdLAogICAgICAgICAgICAgICLkuLvpm4blkIjlh4bnoa7njoc9IiwgdmFsdWVzWyJhY2N1cmFjeV92YWxpZF9pbnB1dHMiXSwKICAgICAgICAgICAgICAiOTDpopjmlY/mhJ/mgKflh4bnoa7njoc9IiwgdmFsdWVzWyJzZW5zaXRpdml0eV9hY2N1cmFjeSJdLAogICAgICAgICAgICAgICLosIPnlKg9IiwgdmFsdWVzWyJhcGlfYXR0ZW1wdHMiXSwKICAgICAgICAgICAgICAidG9rZW49IiwgdmFsdWVzWyJrbm93bl90b3RhbF90b2tlbnMiXSwgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCLntK/orqHosIPnlKjmrKHmlbDvvJoiLCBzdW1tYXJ5WyJjdW11bGF0aXZlX2FwaV9hdHRlbXB0cyJdLCBmbHVzaD1UcnVlKQoKCmRlZiBydW5fZGV2ZWxvcG1lbnQoY2xpZW50KToKICAgICIiIuacgOWkmjI05qyh6LCD55So77yM6aqM6K+BM+S4queLrOeri+W8gOWPkemimOeahOWbm+enjea1geeoi+OAgiIiIgogICAgY29uZmlnID0gRXhwZXJpbWVudENvbmZpZyhtb2RlbD0iZGVlcHNlZWstZmxhc2giLCBtYXhfYXBpX2NhbGxzPTI0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPTc2OCwgbWF4X3Jldmlld19yb3VuZHM9MiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2Vuc2l0aXZpdHlfZXhjbHVkZWRfcm93cz0oKSkKICAgIGVuZ2luZSA9IEV4cGVyaW1lbnRSdW5uZXIoREVWX0RBVEEsIGNsaWVudCwgUk9PVCAvICJkZXZlbG9wbWVudCIsIGNvbmZpZykKICAgIHByaW50KCLlvIDlp4vni6znq4vlvIDlj5Hmo4Dmn6XvvJoz6YGT5Lq65bel5p6E6YCg6aKYIMOXIDTnp43mlrnms5XjgIIiLCBmbHVzaD1UcnVlKQogICAgcmVzdWx0ID0gZW5naW5lLnJ1bigpCiAgICBjb21wYWN0X3N1bW1hcnkocmVzdWx0KQogICAgZm9yIG1ldGhvZCwgc3RhdHMgaW4gcmVzdWx0WyJtZXRob2RzIl0uaXRlbXMoKToKICAgICAgICBpZiBub3Qgc3RhdHNbImFsbF92YWxpZF9pbnB1dHNfZmluaXNoZWQiXSBvciBzZXQoc3RhdHNbInN0YXR1c19jb3VudHMiXSkgIT0geyJjb21wbGV0ZWQifToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie21ldGhvZH0g5a2Y5Zyo6LCD55SoL+agvOW8j+mXrumimO+8jOWFiOS/ruWkjeWGjeWQr+WKqOato+W8j+WunumqjOOAgiIpCiAgICBwcmludCgi5byA5Y+R5qOA5p+l5a6M5oiQ44CC6K+35qOA5p+l5Lul5LiK57uT5p6c77yM5YaN6L+Q6KGM5q2j5byP5a6e6aqM5Y2V5YWD5qC844CCIiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBlbmdpbmUKCgpkZWYgcnVuX2Zvcm1hbChjbGllbnQpOgogICAgIiIi5a+55ZCM5LiAMTAw5p2h5Y6f5aeL6K6w5b2V6aKE5rOo5YaM77yaOTfmnaHkuLvpm4blkIjjgIE5MOadoeaVj+aEn+aAp+mbhuWQiOOAgiIiIgogICAgZGV2ZWxvcG1lbnRfc3VtbWFyeSA9IFJPT1QgLyAiZGV2ZWxvcG1lbnQvc3VtbWFyeS5qc29uIgogICAgaWYgbm90IGRldmVsb3BtZW50X3N1bW1hcnkuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCLor7flhYjlrozmiJDni6znq4vlvIDlj5Hmo4Dmn6XjgIIiKQogICAgZGV2ZWxvcG1lbnQgPSBqc29uLmxvYWRzKGRldmVsb3BtZW50X3N1bW1hcnkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgaWYgbm90IGFsbChzWyJhbGxfdmFsaWRfaW5wdXRzX2ZpbmlzaGVkIl0gYW5kIHNldChzWyJzdGF0dXNfY291bnRzIl0pID09IHsiY29tcGxldGVkIn0KICAgICAgICAgICAgICAgZm9yIHMgaW4gZGV2ZWxvcG1lbnRbIm1ldGhvZHMiXS52YWx1ZXMoKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCLlvIDlj5Hmo4Dmn6Xku43mnInlpLHotKXmiJbmnKrlrozmiJDpobnjgIIiKQogICAgY29uZmlnID0gRXhwZXJpbWVudENvbmZpZyhtb2RlbD0iZGVlcHNlZWstZmxhc2giLCBtYXhfYXBpX2NhbGxzPTgwMCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2Vucz03NjgsIG1heF9yZXZpZXdfcm91bmRzPTIpCiAgICBlbmdpbmUgPSBFeHBlcmltZW50UnVubmVyKERBVEEsIGNsaWVudCwgUk9PVCAvICJmb3JtYWwiLCBjb25maWcpCiAgICBwcmludCgi5q2j5byP6K6+572u5bey5Ya757uT77yaIiwgZW5naW5lLmZpbmdlcnByaW50LCBmbHVzaD1UcnVlKQogICAgcHJpbnQoIuWOn+WnizEwMOadoe+8mzk35p2h5pyJ5paH5pys6YCJ6aG577ybOTDmnaHpooTms6jlhozmlY/mhJ/mgKfpm4blkIjjgIIiLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoIuWbm+e7hOaXoOmHjeivleaXtuacgOWkmjc3NuasoUFQSeiwg+eUqO+8m+mihOeulzgwMOWPqueUqOS6juaYjuehrumcgOimgeeahOaBouWkje+8jOeoi+W6j+S4jemakOaAp+mHjeivleOAgiIsIGZsdXNoPVRydWUpCiAgICBmb3IgbWV0aG9kIGluIE1FVEhPRFM6CiAgICAgICAgcHJpbnQoIlxu5byA5aeL5pa55rOV77yaIiwgbWV0aG9kLCBmbHVzaD1UcnVlKQogICAgICAgIHJlc3VsdCA9IGVuZ2luZS5ydW4obWV0aG9kcz0obWV0aG9kLCkpCiAgICAgICAgY29tcGFjdF9zdW1tYXJ5KHJlc3VsdCkKICAgICAgICBpZiBub3QgcmVzdWx0WyJtZXRob2RzIl1bbWV0aG9kXVsiYWxsX3ZhbGlkX2lucHV0c19maW5pc2hlZCJdOgogICAgICAgICAgICBwcmludCgi5pyJ5pyq5a6M5oiQ6aG577yM5bey5L+d5a2Y5pat54K544CC5pys5qyh5YGc5q2i77yM5LiN5Lyq6YCg5a6M5pW057uT5p6c44CCIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmV0dXJuIGVuZ2luZQogICAgcHJpbnQoIlxu5Zub57uE5q2j5byP6K+E5rWL5bey57uT5p2f44CC5LiL5LiA5q2l5LuO55yf5a6e5pel5b+X55Sf5oiQ5Zu+6KGo44CB5qGI5L6L5LiOTGFUZVjmiqXlkYrjgIIiLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGVuZ2luZQoKCnByaW50KCLlrozmlbTlrp7pqozmqKHlnZflt7LliqDovb3vvJvmraTml7bmsqHmnInosIPnlKjmqKHlnosgQVBJ44CCIikKcHJpbnQoIui/kOihjOmhuuW6j++8mmRldl9lbmdpbmUgPSBydW5fZGV2ZWxvcG1lbnQoX2Fzc2lnbm1lbnQyX2NsaWVudCkiKQpwcmludCgi5qOA5p+l6YCa6L+H5ZCO77yaZm9ybWFsX2VuZ2luZSA9IHJ1bl9mb3JtYWwoX2Fzc2lnbm1lbnQyX2NsaWVudCkiKQo=', 'sha256': '764ea60e0cfd4411a50ea95ce8561a9f0eb38481f4cfad22ae042c87f8f3e15c'}, 'analyze_results.py': {'base64': '...', 'sha256': '51d811d29cf14bc1bc9fda44170f033abbd3239518c9af1a7cad448f795fb17f'}, 'supplemental_runner.py': {'base64': '...', 'sha256': '7461bc155c22c137b375ed14d0f1f15b6279010f0c20b7a66de15738a446add9'}, 'supplemental_protocol.json': {'base64': '...', 'sha256': 'f14d3e403ec7e38b43596c8a6ea14ce36d933f5c1addd12f7abf413a0beee802'}, 'supplemental_items.json': {'base64': 'ewogICJwdXJwb3NlIjogIlBvc3QtaG9jIHByb3RvY29sIHJvYnVzdG5lc3Mgc3R1ZHk7IGluZGVwZW5kZW50bHkgaGFuZC1jb25zdHJ1Y3RlZCBjbG9zZWQtd29ybGQgcXVlc3Rpb25zLCBub3QgcGhhcm1hY2lzdC1rbm93bGVkZ2UgZXZhbHVhdGlvbi4iLAogICJwcm92ZW5hbmNlIjogIuWFqOmDqOS6i+WunuOAgeinhOWImeWSjOaVsOWAvOWcqOmimOW5suS4ree7meWumuOAguacquS7juato+W8jzEwMOmimOOAgeWFtuino+aekOOAgeWOn2Zldy1zaG9056S65L6L5oiW5Y6fM+mBk+W8gOWPkemimOaKveWPluOAgiIsCiAgInNwbGl0X3BvbGljeSI6ICJEMDEtRDA45Li66K6+6K6hL+W8gOWPkeajgOafpe+8jFAwMS1QMDjkuLpoZWxkLW91dCBwcm9iZeOAguS4pOe7hOWPiuWbuuWumnYy5Zyo5Lu75L2V5paw5aKeQVBJ6LCD55So5YmN5Ya757uT77yb5LiN5b6X5qC55o2ucHJvYmXnu5Pmnpzkv67mlLl2MuOAgiIsCiAgIml0ZW1zIjogWwogICAgewogICAgICAiaWQiOiAiRDAxIiwgInNwbGl0IjogImRlc2lnbl9kZXYiLCAicXVlc3Rpb25fdHlwZSI6ICLmnIDkvbPpgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi5pys57uD5Lmg55qE5bqT5a2Y6KeE5YiZ77ya5pyA57uI5bqT5a2Y5Lil5qC85bCR5LqOMzDnm5Lml7bmiY3moIforrDigJzpnIDooaXotKfigJ3jgILotbfliJ3mnIkxOOebku+8jOmaj+WQjuWFpeW6kzIw55uS44CB5Ye65bqTN+ebku+8jOayoeacieWFtuS7luWPmOWMluOAguS4i+WIl+e7k+iuuuWTqumhueato+ehru+8nyIsCiAgICAgICJvcHRpb24iOiB7IkEiOiAi5pyA57uIMjXnm5LvvIzpnIDooaXotKciLCAiQiI6ICLmnIDnu4gzMeebku+8jOmcgOihpei0pyIsICJDIjogIuacgOe7iDMx55uS77yM5LiN6ZyA6KGl6LSnIiwgIkQiOiAi5pyA57uIMzjnm5LvvIzkuI3pnIDooaXotKcifSwKICAgICAgImFuc3dlciI6ICJDIiwgImFuYWx5c2lzIjogIjE4KzIwLTc9MzHvvJszMeS4jeWwj+S6jjMw77yM5Zug5q2k5LiN6ZyA6KGl6LSn44CCIgogICAgfSwKICAgIHsKICAgICAgImlkIjogIkQwMiIsICJzcGxpdCI6ICJkZXNpZ25fZGV2IiwgInF1ZXN0aW9uX3R5cGUiOiAi5pyA5L2z6YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuW3suefpTHlsI/ml7Y9NjDliIbpkp/jgIEx5YiG6ZKfPTYw56eS44CC5LiA5qyh55uY54K555So5pe2MuWwj+aXtjE15YiG6ZKf44CC5Lul5LiL5ZOq6aG56KGo56S655qE5pe26ZW/5LiN562J5LqO5pys5qyh55uY54K555So5pe277yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICIxMzXliIbpkp8iLCAiQiI6ICIyLjI15bCP5pe2IiwgIkMiOiAiODEwMOenkiIsICJEIjogIjEyNeWIhumSnyJ9LAogICAgICAiYW5zd2VyIjogIkQiLCAiYW5hbHlzaXMiOiAiMsOXNjArMTU9MTM15YiG6ZKfPTIuMjXlsI/ml7Y9ODEwMOenku+8mzEyNeWIhumSn+S4jeetieS7t+OAgiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJEMDMiLCAic3BsaXQiOiAiZGVzaWduX2RldiIsICJxdWVzdGlvbl90eXBlIjogIuWkmumhuemAieaLqemimCIsCiAgICAgICJxdWVzdGlvbiI6ICLomZrmnoTnmoTmoLflk4Hlh7rlupPop4TliJnvvJrlvZPliY3ml6XmlbDkuI3otoXov4fmiKrmraLml6XvvIzlubbkuJTlvIDlsIHlkI7nmoTnu4/ov4fml6XmlbDkuI3otoXov4cy77yM5omN5Y+v5Ye65bqT77yb5byA5bCB5b2T5aSp57uP6L+H5pel5pWw5Li6MOOAguW9k+WJjeaYr+esrDMw5pel44CC5LiL5YiX5ZOq5Lqb5qC35ZOB5ZCM5pe256ym5ZCI5Lik5p2h6KeE5YiZ77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICLmiKrmraLnrKwzMOaXpe+8jOesrDI45pel5byA5bCBIiwgIkIiOiAi5oiq5q2i56ysMjnml6XvvIznrKwzMOaXpeW8gOWwgSIsICJDIjogIuaIquatouesrDMx5pel77yM56ysMjfml6XlvIDlsIEiLCAiRCI6ICLmiKrmraLnrKwzMuaXpe+8jOesrDI55pel5byA5bCBIiwgIkUiOiAi5oiq5q2i56ysMzDml6XvvIznrKwzMOaXpeW8gOWwgSJ9LAogICAgICAiYW5zd2VyIjogIkFERSIsICJhbmFseXNpcyI6ICJB57uP6L+HMuaXpeS4lOacqui/h+aIquatouaXpe+8jETnu4/ov4cx5pel77yMRee7j+i/hzDml6XvvJtC5bey6L+H5oiq5q2i5pel77yMQ+e7j+i/hzPml6XjgIIiCiAgICB9LAogICAgewogICAgICAiaWQiOiAiRDA0IiwgInNwbGl0IjogImRlc2lnbl9kZXYiLCAicXVlc3Rpb25fdHlwZSI6ICLphY3kvI3pgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi5LuF5oyJ5Lul5LiL57uZ5a6a6KeE5YiZ6YCJ5oup5om55qyh77ya5o6S6Zmk5bqT5a2Y5Li6MOiAhe+8jOS9meiAheS8mOWFiOmAieaLqeaIquatouaXpeacgOaXqeeahOaJueasoe+8m+aIquatouaXpeebuOWQjOaXtuaMieeUsuOAgeS5meOAgeS4meOAgeS4geeahOmhuuW6j+S8mOWFiOOAgueUsu+8muaIquatouesrDE15pel44CB5bqT5a2YOO+8m+S5me+8muaIquatouesrDEy5pel44CB5bqT5a2YMO+8m+S4me+8muaIquatouesrDE15pel44CB5bqT5a2YNO+8m+S4ge+8muaIquatouesrDE45pel44CB5bqT5a2YOeOAguW6lOmAieWTquaJue+8nyIsCiAgICAgICJvcHRpb24iOiB7IkEiOiAi55SyIiwgIkIiOiAi5LmZIiwgIkMiOiAi5LiZIiwgIkQiOiAi5LiBIn0sCiAgICAgICJhbnN3ZXIiOiAiQSIsICJhbmFseXNpcyI6ICLkuZnml6DlupPlrZjlhYjmjpLpmaTvvJvnlLLlkozkuJnmiKrmraLml6XlkIzkuLoxNe+8jOaMiee7meWumumhuuW6j+mAieeUsuOAgiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJEMDUiLCAic3BsaXQiOiAiZGVzaWduX2RldiIsICJxdWVzdGlvbl90eXBlIjogIuacgOS9s+mAieaLqemimCIsCiAgICAgICJxdWVzdGlvbiI6ICLmnKzpopjljIXoo4Xop4TliJnkuLrmr4/nrrE455uS44CB5q+P55uSMTLooovjgILku5PlupPnjrDmnIky5pW0566x44CB6aKd5aSWNeaVtOebkuWSjOmineWkljfmlaPooovvvIzkuInogIXkupLkuI3ph43lj6DjgILkuIDlhbHlpJrlsJHooovvvJ8iLAogICAgICAib3B0aW9uIjogeyJBIjogIjE5OeiiiyIsICJCIjogIjI1MuiiiyIsICJDIjogIjI1OeiiiyIsICJEIjogIjI2N+iiiyJ9LAogICAgICAiYW5zd2VyIjogIkMiLCAiYW5hbHlzaXMiOiAiMsOXOMOXMTIrNcOXMTIrNz0xOTIrNjArNz0yNTnjgIIiCiAgICB9LAogICAgewogICAgICAiaWQiOiAiRDA2IiwgInNwbGl0IjogImRlc2lnbl9kZXYiLCAicXVlc3Rpb25fdHlwZSI6ICLlpJrpobnpgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi5pys57uD5Lmg6KeE5a6a5YWl6YCJ6K6w5b2V5b+F6aG75ZCM5pe25ruh6Laz77ya5pWw6YeP6Iez5bCRMTDvvIzor7vmlbDkvY3kuo4y6IezOOS5i+mXtO+8iOWMheaLrDLlkow477yJ77yM5YyF6KOF5qCH6K6w5Li64oCc5a6M5pW04oCd44CC5LiL5YiX5ZOq5Lqb6K6w5b2V5YWl6YCJ77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICLmlbDph48xMO+8jOivu+aVsDLvvIzlrozmlbQiLCAiQiI6ICLmlbDph48577yM6K+75pWwNO+8jOWujOaVtCIsICJDIjogIuaVsOmHjzE177yM6K+75pWwOO+8jOWujOaVtCIsICJEIjogIuaVsOmHjzE477yM6K+75pWwNe+8jOegtOaNnyIsICJFIjogIuaVsOmHjzEx77yM6K+75pWwOe+8jOWujOaVtCJ9LAogICAgICAiYW5zd2VyIjogIkFDIiwgImFuYWx5c2lzIjogIkHjgIFD5ZCM5pe25ruh6Laz5LiJ5Liq5p2h5Lu277ybQuaVsOmHj+S4jei2s+OAgUTljIXoo4XkuI3lrozmlbTjgIFF6K+75pWw6LaK55WM44CCIgogICAgfSwKICAgIHsKICAgICAgImlkIjogIkQwNyIsICJzcGxpdCI6ICJkZXNpZ25fZGV2IiwgInF1ZXN0aW9uX3R5cGUiOiAi5pyA5L2z6YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuiusOi0puinhOWIme+8muWFiOaKiuavj+eslOiusOW9leiuoeWFpeeahOaVsOmHj+mZkOWItuS4uuS4jei2hei/hzUw77yM5YaN5oqK5ZCE56yU6K6h5YWl5pWw6YeP55u45Yqg44CC5a6e6ZmF5Lik56yU5pWw6YeP5YiG5Yir5Li6NjXlkowzMOOAguS7peS4i+ivtOazleWTqumhuemUmeivr++8nyIsCiAgICAgICJvcHRpb24iOiB7IkEiOiAi5ZCI6K6h6K6h5YWlODAiLCAiQiI6ICLnrKzkuIDnrJTorqHlhaU1MCIsICJDIjogIuesrOS6jOeslOiuoeWFpTMwIiwgIkQiOiAi5ZCI6K6h6K6h5YWlOTUifSwKICAgICAgImFuc3dlciI6ICJEIiwgImFuYWx5c2lzIjogIumAkOeslOmZkOWItuWQjuS4ujUw5ZKMMzDvvIzlkIjorqE4MO+8mzk15b+955Wl5LqG56ys5LiA56yU5LiK6ZmQ44CCIgogICAgfSwKICAgIHsKICAgICAgImlkIjogIkQwOCIsICJzcGxpdCI6ICJkZXNpZ25fZGV2IiwgInF1ZXN0aW9uX3R5cGUiOiAi5aSa6aG56YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuacrOmimOWumuS5ieKAnOiusOW9leWQiOagvOeOh+KAnT3lkIjmoLzorrDlvZXnrJTmlbAv5YWo6YOo6K6w5b2V56yU5pWw77yM5LiO5ZCE56yU5pWw6YeP5peg5YWz44CC5YWx5pyJNOeslO+8muWQiOagvDEw5Lu244CB5ZCI5qC8MjDku7bjgIHlkIjmoLwzMOS7tuOAgeS4jeWQiOagvDQw5Lu244CC5LiL5YiX6K+05rOV5ZOq5Lqb5q2j56Gu77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICLorrDlvZXlkIjmoLznjofkuLo3NSUiLCAiQiI6ICLlkIjmoLzorrDlvZXlkIjorqE2MOS7tiIsICJDIjogIuaJgOacieiusOW9leWQiOiuoTEwMOS7tiIsICJEIjogIuiusOW9leWQiOagvOeOh+S4ujYwJSIsICJFIjogIuWFqOmDqOiusOW9leWFsTPnrJQifSwKICAgICAgImFuc3dlciI6ICJBQkMiLCAiYW5hbHlzaXMiOiAi5ZCI5qC8My/mgLs0PTc1Je+8m+WQiOagvOaVsOmHjzEwKzIwKzMwPTYw77yM5YWo6YOo5pWw6YePMTAw44CCIgogICAgfSwKICAgIHsKICAgICAgImlkIjogIlAwMSIsICJzcGxpdCI6ICJoZWxkX291dF9wcm9iZSIsICJxdWVzdGlvbl90eXBlIjogIuacgOS9s+mAieaLqemimCIsCiAgICAgICJxdWVzdGlvbiI6ICLmnKzpopjop4TlrprvvJrlj6/liIbphY3mlbDph489bWF4KDAs546w5a2Y5pWw6YePLemihOeVmeaVsOmHjynvvIzmnKzorqLljZXliIbphY3ph4/kuI3lvpfotoXov4flj6/liIbphY3mlbDph4/miJborqLljZXpnIDmsYLjgILnjrDlrZg4NOS7tu+8jOmihOeVmTE55Lu277yM5paw6K6i5Y2V6ZyA5rGCNjjku7bjgILmnIDlpJrlj6/nu5nmnKzorqLljZXliIbphY3lpJrlsJHku7bvvJ8iLAogICAgICAib3B0aW9uIjogeyJBIjogIjY45Lu2IiwgIkIiOiAiNjXku7YiLCAiQyI6ICI4NOS7tiIsICJEIjogIjQ55Lu2In0sCiAgICAgICJhbnN3ZXIiOiAiQiIsICJhbmFseXNpcyI6ICLlj6/liIbphY04NC0xOT02Ne+8m21pbig2NSw2OCk9NjXjgIIiCiAgICB9LAogICAgewogICAgICAiaWQiOiAiUDAyIiwgInNwbGl0IjogImhlbGRfb3V0X3Byb2JlIiwgInF1ZXN0aW9uX3R5cGUiOiAi5aSa6aG56YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuW3suefpTHljYc9MTAwMOavq+WNh+OAgTHmr6vljYc9MTAwMOW+ruWNh+OAguWuueWZqOS4reeahOS9k+enr+aYrzAuMDc15Y2H44CC5Lul5LiL5ZOq5Lqb6KGo56S65LiO6K+l5L2T56ev55u4562J77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICI3Neavq+WNhyIsICJCIjogIjc1MDDlvq7ljYciLCAiQyI6ICI3NTAwMOW+ruWNhyIsICJEIjogIjAuNzXmr6vljYciLCAiRSI6ICI3NTDmr6vljYcifSwKICAgICAgImFuc3dlciI6ICJBQyIsICJhbmFseXNpcyI6ICIwLjA3NcOXMTAwMD03Neavq+WNh++8jOWGjeS5mDEwMDDkuLo3NTAwMOW+ruWNh+OAgiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJQMDMiLCAic3BsaXQiOiAiaGVsZF9vdXRfcHJvYmUiLCAicXVlc3Rpb25fdHlwZSI6ICLmnIDkvbPpgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi5pys6aKY5piO56Gu6KeE5a6a5o2f6ICX546HPeaNn+iAl+S7tuaVsC/ljp/lp4vmgLvku7bmlbDjgILljp/lp4vlhbHmnIkyMDDku7bvvIzlhbbkuK025Lu25o2f6ICX77yM5YW25L2Z5YWo6YOo5L+d55WZ44CC5LiL5YiX6K+05rOV5ZOq6aG55LiN56ym5ZCI57uZ5a6a6KeE5YiZ5oiW5pWw5o2u77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICLmjZ/ogJfnjofkuLozJSIsICJCIjogIuS/neeVmTE5NOS7tiIsICJDIjogIuiuoeeul+aNn+iAl+eOh+eahOWIhuavjeS4ujIwMCIsICJEIjogIuaNn+iAl+eOh+W6lOiuoeeul+S4ujYvMTk0In0sCiAgICAgICJhbnN3ZXIiOiAiRCIsICJhbmFseXNpcyI6ICI2LzIwMD0zJe+8jDIwMC02PTE5NO+8m+S7peS/neeVmeS7tuaVsDE5NOS9nOWIhuavjei/neWPjee7meWumuWumuS5ieOAgiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJQMDQiLCAic3BsaXQiOiAiaGVsZF9vdXRfcHJvYmUiLCAicXVlc3Rpb25fdHlwZSI6ICLphY3kvI3pgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi57yW56CB6KeE5YiZ5oyJ5YWI5ZCO6aG65bqP5omn6KGM77ya6Iul5qCH562+57y65aSx77yM57yW56CB5Li657qi77yb5ZCm5YiZ77yM6Iul6YeN6YeP5YGP5beu5Lil5qC85aSn5LqOMu+8jOe8lueggeS4uum7hO+8m+WFtuS9mee8lueggeS4uuiTneOAguS4gOS7tuagt+WTgeagh+etvuWujOaVtO+8jOmHjemHj+WBj+W3ruaBsOWlveS4ujLjgILlr7nlupTku4DkuYjnvJbnoIHvvJ8iLAogICAgICAib3B0aW9uIjogeyJBIjogIue6oiIsICJCIjogIum7hCIsICJDIjogIuiTnSIsICJEIjogIuaXoOazleagueaNruinhOWImeehruWumiJ9LAogICAgICAiYW5zd2VyIjogIkMiLCAiYW5hbHlzaXMiOiAi5qCH562+5LiN57y65aSx77yb5YGP5beuMuS4jeS4peagvOWkp+S6jjLvvIzlm6DmraTov5vlhaXlhbbkvZnmg4XlhrXvvIznvJbnoIHok53jgIIiCiAgICB9LAogICAgewogICAgICAiaWQiOiAiUDA1IiwgInNwbGl0IjogImhlbGRfb3V0X3Byb2JlIiwgInF1ZXN0aW9uX3R5cGUiOiAi5aSa6aG56YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuaXpeW/l+inhOWIme+8muWQjOS4gOe8luWPt+WkmuasoeWHuueOsOaXtuS7heS/neeVmeacgOWQjuS4gOasoeeahOaVsOmHj+WPmOWMlu+8jOWGjeaxguWHgOWPmOWMluOAguaMiemhuuW6j+aUtuWIsFg6KzXjgIFZOi0y44CBWDorN+OAgVo6KzPjgILku6XkuIvlk6rkupvor7Tms5XmraPnoa7vvJ8iLAogICAgICAib3B0aW9uIjogeyJBIjogIljlupTkv53nlZkrNSIsICJCIjogIljlupTkv53nlZkrNyIsICJDIjogIuacgOe7iOWHgOWPmOWMluS4uis4IiwgIkQiOiAi5Y676YeN5ZCO5pyJM+adoeiusOW9lSIsICJFIjogIuacgOe7iOWHgOWPmOWMluS4uisxMyJ9LAogICAgICAiYW5zd2VyIjogIkJDRCIsICJhbmFseXNpcyI6ICLkv53nlZlYOis344CBWTotMuOAgVo6KzPvvIzlhbEz5p2h77yM5YeA5Y+Y5YyWNy0yKzM9OOOAgiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJQMDYiLCAic3BsaXQiOiAiaGVsZF9vdXRfcHJvYmUiLCAicXVlc3Rpb25fdHlwZSI6ICLmnIDkvbPpgInmi6npopgiLAogICAgICAicXVlc3Rpb24iOiAi5LiA5Liq6L+Q6L6T566x5pyA5aSa6KOFN+ebku+8jOebkuWtkOS4jeiDveaLhuWIhu+8jOS5n+S4jeWFgeiuuOi2hei/h+avj+euseS4iumZkOOAgueOsOWcqOW/hemhu+ijheWujDMx55uS77yM566x5a2Q5pWw6YeP5LiN6ZmQ44CC5pyA5bCR6ZyA6KaB5aSa5bCR566x77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICI1566xIiwgIkIiOiAiNOeusSIsICJDIjogIjbnrrEiLCAiRCI6ICI3566xIn0sCiAgICAgICJhbnN3ZXIiOiAiQSIsICJhbmFseXNpcyI6ICI0566x5pyA5aSaMjjnm5LkuI3otrPvvIw1566x5pyA5aSaMzXnm5LotrPlpJ/vvIzmiYDku6XmnIDlsJE1566x44CCIgogICAgfSwKICAgIHsKICAgICAgImlkIjogIlAwNyIsICJzcGxpdCI6ICJoZWxkX291dF9wcm9iZSIsICJxdWVzdGlvbl90eXBlIjogIuWkmumhuemAieaLqemimCIsCiAgICAgICJxdWVzdGlvbiI6ICLmnKzpopjliIbphY3op4TliJnvvJrlhYHorrjpg6jliIbmu6HotrPorqLljZXvvIzmjInpq5jjgIHkuK3jgIHkvY7kvJjlhYjnuqfkvp3mrKHliIbphY3vvIzmr4/nuqflhYjlsL3ph4/mu6HotrPpnIDmsYLvvIzlho3miorliankvZnmlbDph4/nu5nkuIvkuIDnuqfvvJvmgLvliIbphY3kuI3og73otoXov4fnjrDmnIkyMOS7tuOAgumrmOOAgeS4reOAgeS9jumcgOaxguS+neasoeS4ujEy44CBMTHjgIE05Lu244CC5LiL5YiX5ZOq5Lqb57uT6K665q2j56Gu77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICLpq5jkvJjlhYjnuqfliIbliLAxMuS7tiIsICJCIjogIuS4reS8mOWFiOe6p+WIhuWIsDEx5Lu2IiwgIkMiOiAi5L2O5LyY5YWI57qn5YiG5YiwMOS7tiIsICJEIjogIuWIhumFjeWujOaIkOWQjuWJqeS9mTDku7YiLCAiRSI6ICLlrp7pmYXlhbHliIbphY0yN+S7tiJ9LAogICAgICAiYW5zd2VyIjogIkFDRCIsICJhbmFseXNpcyI6ICLpq5jlhYjlj5YxMu+8jOWJqTjnu5nkuK3vvIzkvY7ml6DliankvZnvvJvmgLvliIbphY0yMO+8jOS9meminTDjgIIiCiAgICB9LAogICAgewogICAgICAiaWQiOiAiUDA4IiwgInNwbGl0IjogImhlbGRfb3V0X3Byb2JlIiwgInF1ZXN0aW9uX3R5cGUiOiAi5pyA5L2z6YCJ5oup6aKYIiwKICAgICAgInF1ZXN0aW9uIjogIuiZmuaehOeahOWIhuautemHh+i0reiuoeS7t+inhOWIme+8muWJjTEw55uS5q+P55uSOeWFg++8jOi2hei/hzEw55uS55qE6YOo5YiG5q+P55uSN+WFg++8m+ayoeacieWFtuS7lui0ueeUqOaIluaKmOaJo+OAgui0reS5sDE055uS5bqU5pSv5LuY5aSa5bCR5YWD77yfIiwKICAgICAgIm9wdGlvbiI6IHsiQSI6ICIxMjblhYMiLCAiQiI6ICIxMTjlhYMiLCAiQyI6ICI5OOWFgyIsICJEIjogIjEwMOWFgyJ9LAogICAgICAiYW5zd2VyIjogIkIiLCAiYW5hbHlzaXMiOiAi5YmNMTDnm5I5MOWFg++8jOmineWkljTnm5IyOOWFg++8jOWQiOiuoTExOOWFg+OAgiIKICAgIH0KICBdCn0K', 'sha256': '96e12e9957e0c43646c4e886c691039016c949693f200984be2805ad2dc0626b'}, 'analyze_supplemental.py': {'base64': 'IiIi6KGl5YWF5a6e6aqM5Y+q6K+75aSN566X77ya5LiN5p6E6YCgQVBJ5a6i5oi356uv77yM5LiN5pS55Ya757uT5rqQ56CB5oiW5Y6f5aeL5pel5b+X44CCIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHN1cHBsZW1lbnRhbF9ydW5uZXIgYXMgc3IKCgpkZWYgbG9hZF9zdXBwbGVtZW50YWwocnVuX2RpciwgZm9ybWFsX2RpciwgcHJvdG9jb2xfcGF0aD1Ob25lKToKICAgICIiIuaMieW3suWGu+e7k+WNj+iuruWSjOmAkOiwg+eUqGxlZGdlcumHjeaWsOivhOWIhu+8jOWQjOaXtumqjOivgeeOsOaciea0vueUn+e7k+aenOOAgiIiIgogICAgZGlyZWN0b3J5ID0gUGF0aChydW5fZGlyKQogICAgcHJvdG9jb2xfcGF0aCA9IFBhdGgocHJvdG9jb2xfcGF0aCkgaWYgcHJvdG9jb2xfcGF0aCBlbHNlIGRpcmVjdG9yeSAvICJwcm90b2NvbC5qc29uIgogICAgcHJvdG9jb2wsIHNvdXJjZSA9IHNyLmxvYWRfcHJvdG9jb2wocHJvdG9jb2xfcGF0aCwgZm9ybWFsX2RpcikKICAgIHByb3RvY29sX2NvcHkgPSBqc29uLmxvYWRzKChkaXJlY3RvcnkgLyAicHJvdG9jb2wuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIGlmIHByb3RvY29sX2NvcHkgIT0gcHJvdG9jb2w6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi6KGl5YWF6L+Q6KGM55uu5b2V5Lit55qE5Y2P6K6u5LiN5piv5oyH5a6a5Ya757uT5Y2P6K6u44CCIikKICAgICMg5LuF5Yip55So57qv6K+E5YiG5pa55rOV77yb5Yi75oSP5LiN6LCD55So5Lya5Yib5bu655uu5b2VL+mUgS/lr7zlh7rmlofku7bnmoTliJ3lp4vljJblmajjgIIKICAgIGVuZ2luZSA9IHNyLlN1cHBsZW1lbnRhbFJ1bi5fX25ld19fKHNyLlN1cHBsZW1lbnRhbFJ1bikKICAgIGVuZ2luZS5wcm90b2NvbCwgZW5naW5lLnNvdXJjZSA9IHByb3RvY29sLCBzb3VyY2UKICAgIGVuZ2luZS5qb2JzID0gc3IubWFrZV9qb2JzKHByb3RvY29sLCBzb3VyY2UpCiAgICBlbmdpbmUub3V0cHV0LCBlbmdpbmUubGVkZ2VyID0gZGlyZWN0b3J5LCBkaXJlY3RvcnkgLyAibGVkZ2VyLmpzb25sIgogICAgZW5naW5lLmZpbmdlcnByaW50ID0gcHJvdG9jb2xbImZpbmdlcnByaW50Il0KICAgIGVuZ2luZS5tYXhfYXBpX2NhbGxzID0gcHJvdG9jb2xbInByb3RvY29sIl1bIm1heF9hcGlfY2FsbHMiXQogICAgZW5naW5lLmNsaWVudCA9IE5vbmUKICAgIGVuZ2luZS5sb2FkKCkKICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcygoZGlyZWN0b3J5IC8gImpvYl9tYW5pZmVzdC5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgZXhwZWN0ZWRfbWFuaWZlc3QgPSB7CiAgICAgICAgImZpbmdlcnByaW50IjogZW5naW5lLmZpbmdlcnByaW50LAogICAgICAgICJvcmRlcmVkX2pvYl9pZHMiOiBbam9iWyJqb2JfaWQiXSBmb3Igam9iIGluIGVuZ2luZS5qb2JzXSwKICAgICAgICAicmVxdWVzdF9zaGEyNTYiOiB7am9iWyJqb2JfaWQiXTogc3IuZGlnZXN0KGpvYlsicmVxdWVzdCJdKSBmb3Igam9iIGluIGVuZ2luZS5qb2JzfSwKICAgICAgICAicGxhbm5lZF9jYWxscyI6IDIyMiwKICAgICAgICAiY29tbW9uX2ludmFsaWRfaW5pdGlhbF9yb3dzIjogWzE0LCA2N10sCiAgICB9CiAgICBpZiBtYW5pZmVzdCAhPSBleHBlY3RlZF9tYW5pZmVzdDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLooaXlhYXor7fmsYJtYW5pZmVzdOS4juWGu+e7k+iuoeWIkuS4jeS4gOiHtOOAgiIpCiAgICBvcmRlcmVkX2pvYnMsIHNlZW5faWRzLCBzZWVuX2F0dGVtcHRzID0gW10sIHNldCgpLCBzZXQoKQogICAgZm9yIGV2ZW50IGluIGVuZ2luZS5ldmVudHM6CiAgICAgICAgaWYgZXZlbnRbImV2ZW50Il0gPT0gImNhbGxfc3RhcnQiOgogICAgICAgICAgICBqb2JfaWQgPSBldmVudFsiam9iX2lkIl0KICAgICAgICAgICAgaWYgam9iX2lkIGluIHNlZW5faWRzIG9yIGV2ZW50LmdldCgicmV0cnlfb2YiKSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIumihOazqOWGjOihpeWFheWunumqjOS4jeWFgeiuuOS7u+S9lemHjeWkjeWwneivleOAgiIpCiAgICAgICAgICAgIGlmIHNyLmRpZ2VzdChldmVudFsicmVxdWVzdCJdKSAhPSBldmVudFsicmVxdWVzdF9zaGEyNTYiXSBvciBldmVudFsicmVxdWVzdF9zaGEyNTYiXSAhPSBtYW5pZmVzdFsicmVxdWVzdF9zaGEyNTYiXVtqb2JfaWRdOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi5a6e6ZmF6K+35rGC5LiN562J5LqO5Ya757uT6K+35rGC44CCIikKICAgICAgICAgICAgc2Vlbl9pZHMuYWRkKGpvYl9pZCkKICAgICAgICAgICAgc2Vlbl9hdHRlbXB0cy5hZGQoZXZlbnRbImF0dGVtcHRfaWQiXSkKICAgICAgICAgICAgb3JkZXJlZF9qb2JzLmFwcGVuZChqb2JfaWQpCiAgICAgICAgZWxpZiBldmVudFsiZXZlbnQiXSA9PSAiY2FsbF9maW5pc2giOgogICAgICAgICAgICBpZiBldmVudFsiYXR0ZW1wdF9pZCJdIG5vdCBpbiBzZWVuX2F0dGVtcHRzOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi6LCD55SoZmluaXNo5Ye6546w5Zyo55u45bqUc3RhcnTkuYvliY3jgIIiKQogICAgICAgICAgICBlbGFwc2VkID0gZXZlbnQuZ2V0KCJlbGFwc2VkX3NlY29uZHMiKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShlbGFwc2VkLCAoaW50LCBmbG9hdCkpIG9yIGlzaW5zdGFuY2UoZWxhcHNlZCwgYm9vbCkgb3Igbm90IG1hdGguaXNmaW5pdGUoZWxhcHNlZCkgb3IgZWxhcHNlZCA8IDA6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLosIPnlKjogJfml7bkuI3lkIjms5XjgIIiKQogICAgICAgICAgICBpZiBldmVudC5nZXQoInN0YXR1cyIpIG5vdCBpbiB7Im9rIiwgImFwaV9lcnJvciJ9OgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi5pyq55+l6LCD55So54q25oCB44CCIikKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLooaXlhYVsZWRnZXLlkKvmnKrnn6Xkuovku7bjgIIiKQogICAgaWYgb3JkZXJlZF9qb2JzICE9IG1hbmlmZXN0WyJvcmRlcmVkX2pvYl9pZHMiXVs6bGVuKG9yZGVyZWRfam9icyldOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuWunumZheiwg+eUqOmhuuW6j+S4jeetieS6juWGu+e7k+iuoeWIkueahOWJjee8gOOAgiIpCiAgICBpZiBsZW4ob3JkZXJlZF9qb2JzKSA+IDIyMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLooaXlhYXosIPnlKjmlbDph4/otoXov4fkuIDmrKHmgKforqHliJLjgIIiKQogICAgYWN0dWFsX3Jvd3MgPSBzci5yZWFkX2pzb25sKGRpcmVjdG9yeSAvICJyZXN1bHRzLmpzb25sIikKICAgIGlmIGFjdHVhbF9yb3dzICE9IGVuZ2luZS5yb3dzKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi6KGl5YWF6YCQ6aKY57uT5p6c5LiObGVkZ2Vy6YeN5paw6K+E5YiG5LiN5LiA6Ie044CCIikKICAgIHJlY29yZGVkX3N1bW1hcnkgPSBqc29uLmxvYWRzKChkaXJlY3RvcnkgLyAic3VtbWFyeS5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgcmVjb3JkZWRfYnVkZ2V0ID0gcmVjb3JkZWRfc3VtbWFyeS5nZXQoIm1heF9hcGlfY2FsbHMiKQogICAgaWYgKG5vdCBpc2luc3RhbmNlKHJlY29yZGVkX2J1ZGdldCwgaW50KSBvciBpc2luc3RhbmNlKHJlY29yZGVkX2J1ZGdldCwgYm9vbCkKICAgICAgICAgICAgb3Igbm90IGxlbihvcmRlcmVkX2pvYnMpIDw9IHJlY29yZGVkX2J1ZGdldCA8PSBwcm90b2NvbFsicHJvdG9jb2wiXVsibWF4X2FwaV9jYWxscyJdKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLorrDlvZXnmoTooaXlhYXpooTnrpfkuI3lkIjms5XmiJblrp7pmYXlsJ3or5XmrKHmlbDlt7LotornlYzjgIIiKQogICAgZW5naW5lLm1heF9hcGlfY2FsbHMgPSByZWNvcmRlZF9idWRnZXQKICAgIGFjdHVhbF9zdW1tYXJ5ID0gZW5naW5lLnN1bW1hcnkoKQogICAgZm9yIGtleSBpbiAoImdlbmVyYXRlZF9hdF91dGMiLCk6CiAgICAgICAgcmVjb3JkZWRfc3VtbWFyeS5wb3Aoa2V5LCBOb25lKQogICAgICAgIGFjdHVhbF9zdW1tYXJ5LnBvcChrZXksIE5vbmUpCiAgICBpZiByZWNvcmRlZF9zdW1tYXJ5ICE9IGFjdHVhbF9zdW1tYXJ5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuihpeWFheaxh+aAu+S4jmxlZGdlcumHjeaWsOe7n+iuoeS4jeS4gOiHtOOAgiIpCiAgICByZXR1cm4gZW5naW5lCgoKZGVmIHdyaXRlX291dHB1dHMoZW5naW5lLCBvdXRwdXRfZGlyKToKICAgICIiIuWPquWGmeaMh+Wumua0vueUn+WIhuaekOebruW9le+8m+S4jeS/ruaUuei+k+WFpeebruW9leWGheW3suaciXN1bW1hcnkvcmVzdWx0cy9sZWRnZXLjgIIiIiIKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0X2RpcikucmVzb2x2ZSgpCiAgICBpZiBvdXRwdXQgPT0gZW5naW5lLm91dHB1dC5yZXNvbHZlKCkgb3Igb3V0cHV0ID09IGVuZ2luZS5vdXRwdXQucGFyZW50LnJlc29sdmUoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLliIbmnpDnm67lvZXpobvni6znq4vvvIzkuI3og73nm7TmjqXopobnm5booaXlhYXljp/lp4vovpPlh7rnm67lvZXjgIIiKQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHN1bW1hcnkgPSBlbmdpbmUuc3VtbWFyeSgpCiAgICBzdW1tYXJ5WyJ2YWxpZGF0aW9uIl0gPSAiZnJvemVuX3Byb3RvY29sX3JlcXVlc3RzX2xlZGdlcl9yZXN1bHRzX2FuZF9zdW1tYXJ5X3Bhc3NlZCIKICAgIHN1bW1hcnlbImFuYWx5c2lzX21vZGUiXSA9ICJvZmZsaW5lX25vX0FQSSIKICAgIHNyLmF0b21pY19qc29uKG91dHB1dCAvICJhbmFseXNpcy5qc29uIiwgc3VtbWFyeSkKICAgIHJvd3MgPSBbXQogICAgZm9yIHN0dWR5LCBjb25kaXRpb25zIGluICgoIkEiLCBzci5BX0NPTkRJVElPTlMpLCAoIkIiLCBzci5CX0NPTkRJVElPTlMpKToKICAgICAgICBmb3IgY29uZGl0aW9uIGluIGNvbmRpdGlvbnM6CiAgICAgICAgICAgIGZvciBzdWJzZXQsIG1ldHJpYyBpbiBzdW1tYXJ5WyJzdHVkeV8iICsgc3R1ZHldW2NvbmRpdGlvbl0uaXRlbXMoKToKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsic3R1ZHkiOiBzdHVkeSwgImNvbmRpdGlvbiI6IGNvbmRpdGlvbiwgInN1YnNldCI6IHN1YnNldCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAqKntrZXk6IG1ldHJpY1trZXldIGZvciBrZXkgaW4gKCJuIiwgImNvcnJlY3QiLCAiZm9ybWF0X3ZhbGlkIiwgImFjY3VyYWN5IiwgImZvcm1hdF92YWxpZF9yYXRlIiwgImFsbF9maW5pc2hlZCIsICJhcGlfYXR0ZW1wdHMiLCAia25vd25fdG90YWxfdG9rZW5zIiwgImNhbGxzX3dpdGhfdW5rbm93bl91c2FnZSIpfX0pCiAgICB3aXRoIChvdXRwdXQgLyAibWV0cmljcy5jc3YiKS5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIGhhbmRsZToKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihoYW5kbGUsIGZpZWxkbmFtZXM9bGlzdChyb3dzWzBdKSkKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIHNyLmF0b21pY19qc29uKG91dHB1dCAvICJwYWlyZWRfY29tcGFyaXNvbnMuanNvbiIsIHN1bW1hcnlbInN1cHBsZW1lbnRhcnlfcHJpbWFyeV9jb21wYXJpc29ucyJdKQogICAgcmV0dXJuIHN1bW1hcnkKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcnVuLWRpciIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9ybWFsLWRpciIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHJvdG9jb2wiLCB0eXBlPVBhdGgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCByZXF1aXJlZD1UcnVlLCB0eXBlPVBhdGgpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoYXJndikKICAgIGVuZ2luZSA9IGxvYWRfc3VwcGxlbWVudGFsKGFyZ3MucnVuX2RpciwgYXJncy5mb3JtYWxfZGlyLCBhcmdzLnByb3RvY29sKQogICAgc3VtbWFyeSA9IHdyaXRlX291dHB1dHMoZW5naW5lLCBhcmdzLm91dHB1dF9kaXIpCiAgICBwcmludCgi6KGl5YWF56a757q/5aSN566X5a6M5oiQ77yb5YWo6YOo5Lu75Yqh5pyJ57uI57uT6K6w5b2V77yaIiwgc3VtbWFyeVsiYWxsX2ZpbmlzaGVkIl0pCiAgICBwcmludCgiQVBJ5bCd6K+V5qyh5pWw77yaIiwgc3VtbWFyeVsiYXBpX2F0dGVtcHRzIl0sICLvvJvmnKzmrKHlpI3nrpdBUEnosIPnlKjvvJowIikKICAgIHByaW50KCLovpPlh7rvvJoiLCBhcmdzLm91dHB1dF9kaXIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK', 'sha256': '6685611965392b3c2fdbc0d76c3fcca54454f57dca052e952c4010d25f2e0aa0'}, 'test_supplemental_runner.py': {'base64': 'IiIi6KGl5YWF5a6e6aqM55qE56a757q/5Zue5b2S77yb5LuF5L2/55So5YGH55qE5a6i5oi356uv77yM57ud5LiN6LCD55So572R57uc44CCIiIiCmltcG9ydCBqc29uCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQoKaW1wb3J0IHN1cHBsZW1lbnRhbF9ydW5uZXIgYXMgc3IKCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApGT1JNQUwgPSBST09UIC8gImFjdHVhbF9ydW4vdGFzazFfYXNzaWdubWVudC9mb3JtYWwiCmlmIG5vdCBGT1JNQUwuaXNfZGlyKCk6CiAgICBGT1JNQUwgPSBST09ULnBhcmVudCAvICJyZXN1bHRzL2Zvcm1hbCIKaWYgbm90IEZPUk1BTC5pc19kaXIoKToKICAgIEZPUk1BTCA9IFJPT1QgLyAidGFzazFfYXNzaWdubWVudC9mb3JtYWwiClBST1RPQ09MID0gUk9PVCAvICJzdXBwbGVtZW50YWxfcHJvdG9jb2xfY2FuZGlkYXRlLmpzb24iCmlmIG5vdCBQUk9UT0NPTC5leGlzdHMoKToKICAgIFBST1RPQ09MID0gUk9PVCAvICJzdXBwbGVtZW50YWxfcHJvdG9jb2wuanNvbiIKVEVNUF9ST09UID0gUk9PVCAvICIudGVzdF90bXAiClRFTVBfUk9PVC5ta2RpcihleGlzdF9vaz1UcnVlKQoKCmNsYXNzIE1vY2tDbGllbnQ6CiAgICBtYXhfcmV0cmllcyA9IDAKICAgIGJhc2VfdXJsID0gImh0dHBzOi8vYXBpLmRlZXBzZWVrLmNvbSIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgam9icywgYmVoYXZpb3I9ImNvcnJlY3QiKToKICAgICAgICBzZWxmLmNoYXQgPSBTaW1wbGVOYW1lc3BhY2UoY29tcGxldGlvbnM9c2VsZikKICAgICAgICBzZWxmLmJ5X3JlcXVlc3QgPSB7c3IuZGlnZXN0KGpvYlsicmVxdWVzdCJdKTogam9iIGZvciBqb2IgaW4gam9ic30KICAgICAgICBzZWxmLmNhbGxzID0gW10KICAgICAgICBzZWxmLmJlaGF2aW9yID0gYmVoYXZpb3IKCiAgICBkZWYgY3JlYXRlKHNlbGYsICoqcmVxdWVzdCk6CiAgICAgICAgc2VsZi5jYWxscy5hcHBlbmQocmVxdWVzdCkKICAgICAgICBpZiBzZWxmLmJlaGF2aW9yID09ICJlcnJvciI6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiU0VDUkVUX1ZBTFVFX1NIT1VMRF9OT1RfQkVfTE9HR0VEIikKICAgICAgICBqb2IgPSBzZWxmLmJ5X3JlcXVlc3Rbc3IuZGlnZXN0KHJlcXVlc3QpXQogICAgICAgIG9iaiA9IHsiYW5zd2VyIjogbGlzdChqb2JbInJlZmVyZW5jZSJdKSwgImJyaWVmX3JlYXNvbiI6ICLnprvnur/mtYvor5XlgYflk43lupTjgIIifQogICAgICAgIGlmIGpvYlsic3R1ZHkiXSA9PSAiQiI6CiAgICAgICAgICAgIG9iai51cGRhdGUoYWN0aW9uPSJyZXZpc2UiLCBkZWNpc2lvbl9yZWFzb249Iuemu+e6v+a1i+ivleWGs+etluOAgiIpCiAgICAgICAgY29udGVudCA9IGpzb24uZHVtcHMob2JqKSBpZiBzZWxmLmJlaGF2aW9yICE9ICJtYWxmb3JtZWQiIGVsc2UgInticm9rZW4iCiAgICAgICAgcmV0dXJuIFNpbXBsZU5hbWVzcGFjZSgKICAgICAgICAgICAgY2hvaWNlcz1bU2ltcGxlTmFtZXNwYWNlKG1lc3NhZ2U9U2ltcGxlTmFtZXNwYWNlKGNvbnRlbnQ9Y29udGVudCksIGZpbmlzaF9yZWFzb249InN0b3AiKV0sCiAgICAgICAgICAgIG1vZGVsPSJkZWVwc2Vlay1mbGFzaCIsCiAgICAgICAgICAgIHVzYWdlPVNpbXBsZU5hbWVzcGFjZShwcm9tcHRfdG9rZW5zPTEwLCBjb21wbGV0aW9uX3Rva2Vucz01LCB0b3RhbF90b2tlbnM9MTUpLAogICAgICAgICkKCgpjbGFzcyBTdXBwbGVtZW50YWxUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBzZXRVcENsYXNzKGNscyk6CiAgICAgICAgY2xzLnByb3RvY29sLCBjbHMuc291cmNlID0gc3IubG9hZF9wcm90b2NvbChQUk9UT0NPTCwgRk9STUFMKQogICAgICAgIGNscy5qb2JzID0gc3IubWFrZV9qb2JzKGNscy5wcm90b2NvbCwgY2xzLnNvdXJjZSkKICAgICAgICBjbHMuc291cmNlX2hhc2hlcyA9IHtwOiBzci5maWxlX2RpZ2VzdChGT1JNQUwgLyBwKSBmb3IgcCBpbiBjbHMuc291cmNlWyJoYXNoZXMiXX0KCiAgICBkZWYgbWFrZV9lbmdpbmUoc2VsZiwgZGlyZWN0b3J5LCBjbGllbnQ9Tm9uZSwgYnVkZ2V0PTI0MCk6CiAgICAgICAgcmV0dXJuIHNyLlN1cHBsZW1lbnRhbFJ1bihjbGllbnQgb3IgTW9ja0NsaWVudChzZWxmLmpvYnMpLCBQUk9UT0NPTCwgRk9STUFMLCBkaXJlY3RvcnksIGJ1ZGdldCkKCiAgICBkZWYgdGVzdF9mcm96ZW5fY291bnRzX2FuZF9nb2xkX2luZGVwZW5kZW50X2VsaWdpYmlsaXR5KHNlbGYpOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNlbGYuam9icyksIDIyMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bShqWyJzdHVkeSJdID09ICJBIiBmb3IgaiBpbiBzZWxmLmpvYnMpLCAzMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihzZWxmLnNvdXJjZVsiY2FzZXMiXSksIDk1KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW3hbInJvd19udW1iZXIiXSBmb3IgeCBpbiBzZWxmLnNvdXJjZVsiaW52YWxpZCJdXSwgWzE0LCA2N10pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oe2pbImpvYl9pZCJdIGZvciBqIGluIHNlbGYuam9ic30pLCAyMjIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZWxmLnByb3RvY29sWyJwcm90b2NvbCJdWyJjb25kaXRpb25fb3JkZXJfc2VlZCJdLCA2MDUyMDIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzci5tYWtlX2pvYnMoc2VsZi5wcm90b2NvbCwgc2VsZi5zb3VyY2UpLCBzZWxmLmpvYnMpCiAgICAgICAgZm9yIHN0dWR5LCBrZXkgaW4gKCgiQSIsICJzdHVkeV9BIiksICgiQiIsICJzdHVkeV9CIikpOgogICAgICAgICAgICBvcmRlcnMgPSBsaXN0KHNlbGYucHJvdG9jb2xbInByb3RvY29sIl1ba2V5XVsiY29uZGl0aW9uX29yZGVyX2J5X3VuaXQiXS52YWx1ZXMoKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKGxlbih7dHVwbGUob3JkZXIpIGZvciBvcmRlciBpbiBvcmRlcnN9KSwgMSkKCiAgICBkZWYgdGVzdF9yZXF1ZXN0X2lzb2xhdGlvbl9hbmRfb25seV9mZWVkYmFja19pbnRlcnZlbnRpb24oc2VsZik6CiAgICAgICAgZm9yIGl0ZW0gaW4gc2VsZi5wcm90b2NvbFsicHJvdG9jb2wiXVsic3ludGhldGljX2l0ZW1zIl1bIml0ZW1zIl06CiAgICAgICAgICAgIGZpcnN0LCBzZWNvbmQgPSBzci5hX21lc3NhZ2VzKCJjb3RfdjEiLCBpdGVtKSwgc3IuYV9tZXNzYWdlcygiY290X3YyIiwgaXRlbSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmaXJzdCwgc3IuZnJvemVuLmFuc3dlcl9tZXNzYWdlcygiY290Iiwgc3IuZnJvemVuLnF1ZXN0aW9uX2lucHV0KGl0ZW0pKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmaXJzdFsxOl0sIHNlY29uZFsxOl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KGpzb24ubG9hZHMoZmlyc3RbMV1bImNvbnRlbnQiXSkpLCB7InF1ZXN0aW9uIiwgIm9wdGlvbnMiLCAicXVlc3Rpb25fdHlwZSJ9KQogICAgICAgIGZvciBjYXNlIGluIHNlbGYuc291cmNlWyJjYXNlcyJdOgogICAgICAgICAgICBmaXJzdCwgc2Vjb25kID0gc3IuYl9tZXNzYWdlcygiZmVlZGJhY2siLCBjYXNlKSwgc3IuYl9tZXNzYWdlcygibnVsbF9mZWVkYmFjayIsIGNhc2UpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3RbMF0sIHNlY29uZFswXSkKICAgICAgICAgICAgYSwgYiA9IGpzb24ubG9hZHMoZmlyc3RbMV1bImNvbnRlbnQiXSksIGpzb24ubG9hZHMoc2Vjb25kWzFdWyJjb250ZW50Il0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYS5wb3AoImZlZWRiYWNrIiksIGNhc2VbImZlZWRiYWNrIl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SXNOb25lKGIucG9wKCJmZWVkYmFjayIpKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGEsIGIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYVsicmVtYWluaW5nX3Jldmlld19yb3VuZHMiXSwgMCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChhWyJyZXZpZXdfcm91bmQiXSwgMSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZXQoYVsicXVlc3Rpb24iXSksIHsicXVlc3Rpb24iLCAib3B0aW9ucyIsICJxdWVzdGlvbl90eXBlIn0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KGFbImNhbmRpZGF0ZSJdKSwgeyJhbnN3ZXIiLCAiYnJpZWZfcmVhc29uIn0pCiAgICAgICAgZm9yIGpvYiBpbiBzZWxmLmpvYnM6CiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KGpvYlsicmVxdWVzdCJdKSwgc2V0KHNyLlNFVFRJTkdTKSB8IHsibWVzc2FnZXMifSkKCiAgICBkZWYgdGVzdF9jb21wbGV0ZV9tb2NrX3J1bl9kZW5vbWluYXRvcnNfYWNjb3VudGluZ19hbmRfcmVzdW1lKHNlbGYpOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KGRpcj1URU1QX1JPT1QpIGFzIGRpcmVjdG9yeToKICAgICAgICAgICAgY2xpZW50ID0gTW9ja0NsaWVudChzZWxmLmpvYnMpCiAgICAgICAgICAgIGVuZ2luZSA9IHNlbGYubWFrZV9lbmdpbmUoZGlyZWN0b3J5LCBjbGllbnQpCiAgICAgICAgICAgIHN1bW1hcnkgPSBlbmdpbmUucnVuKHZlcmJvc2U9RmFsc2UpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShzdW1tYXJ5WyJhbGxfZmluaXNoZWQiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oY2xpZW50LmNhbGxzKSwgMjIyKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihlbmdpbmUucm93cygpKSwgMjI2KQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcnlbInRvdGFscyJdWyJrbm93bl90b3RhbF90b2tlbnMiXSwgMzMzMCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW1tYXJ5WyJ0b3RhbHMiXVsiY2FsbHNfd2l0aF91bmtub3duX3VzYWdlIl0sIDApCiAgICAgICAgICAgIGZvciBjb25kaXRpb24gaW4gc3IuQV9DT05ESVRJT05TOgogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW1tYXJ5WyJzdHVkeV9BIl1bY29uZGl0aW9uXVsiYWxsXzE2Il1bImNvcnJlY3QiXSwgMTYpCiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcnlbInN0dWR5X0EiXVtjb25kaXRpb25dWyJoZWxkX291dF9wcm9iZSJdWyJuIl0sIDgpCiAgICAgICAgICAgIGZvciBjb25kaXRpb24gaW4gc3IuQl9DT05ESVRJT05TOgogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW1tYXJ5WyJzdHVkeV9CIl1bY29uZGl0aW9uXVsiY29uZGl0aW9uYWxfOTUiXVsiYWNjdXJhY3kiXSwgMSkKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3VtbWFyeVsic3R1ZHlfQiJdW2NvbmRpdGlvbl1bIm92ZXJhbGxfOTciXVsiYWNjdXJhY3kiXSwgOTUgLyA5NykKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3VtbWFyeVsic3R1ZHlfQiJdW2NvbmRpdGlvbl1bIm92ZXJhbGxfOTciXVsic3RhdHVzX2NvdW50cyJdWyJjb21tb25faW5pdGlhbF9mYWlsdXJlIl0sIDIpCiAgICAgICAgICAgIHByaW1hcnkgPSBzdW1tYXJ5WyJzdXBwbGVtZW50YXJ5X3ByaW1hcnlfY29tcGFyaXNvbnMiXQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNldChwcmltYXJ5KSwgeyJBX2FsbF8xNiIsICJCX2NvbmRpdGlvbmFsXzk1In0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJpbWFyeVsiQV9hbGxfMTYiXVsibiJdLCAxNikKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcmltYXJ5WyJCX2NvbmRpdGlvbmFsXzk1Il1bIm4iXSwgOTUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwodlsiaG9sbV9hZGp1c3RlZF9wX3R3b19jb21wYXJpc29uX2ZhbWlseSJdID09IDEgZm9yIHYgaW4gcHJpbWFyeS52YWx1ZXMoKSkpCiAgICAgICAgICAgIHJlc3VtZWQgPSBzZWxmLm1ha2VfZW5naW5lKGRpcmVjdG9yeSwgY2xpZW50KQogICAgICAgICAgICByZXN1bWVkLnJ1bih2ZXJib3NlPUZhbHNlKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihjbGllbnQuY2FsbHMpLCAyMjIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoe3A6IHNyLmZpbGVfZGlnZXN0KEZPUk1BTCAvIHApIGZvciBwIGluIHNlbGYuc291cmNlX2hhc2hlc30sIHNlbGYuc291cmNlX2hhc2hlcykKCiAgICBkZWYgdGVzdF96ZXJvX2J1ZGdldF9hbmRfbm9fcmV0cnlfb25fZXJyb3Jfb3JfcGFyc2VfZmFpbHVyZShzZWxmKToKICAgICAgICBmb3IgYmVoYXZpb3IsIHN0YXR1cyBpbiAoKCJlcnJvciIsICJhcGlfZXJyb3IiKSwgKCJtYWxmb3JtZWQiLCAicGFyc2VfZXJyb3IiKSk6CiAgICAgICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KGRpcj1URU1QX1JPT1QpIGFzIGRpcmVjdG9yeToKICAgICAgICAgICAgICAgIGNsaWVudCA9IE1vY2tDbGllbnQoc2VsZi5qb2JzLCBiZWhhdmlvcikKICAgICAgICAgICAgICAgIGVuZ2luZSA9IHNlbGYubWFrZV9lbmdpbmUoZGlyZWN0b3J5LCBjbGllbnQsIGJ1ZGdldD0wKQogICAgICAgICAgICAgICAgZW5naW5lLnJ1bih2ZXJib3NlPUZhbHNlKQogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChjbGllbnQuY2FsbHMsIFtdKQogICAgICAgICAgICAgICAgZW5naW5lLm1heF9hcGlfY2FsbHMgPSAxCiAgICAgICAgICAgICAgICBlbmdpbmUucnVuKHZlcmJvc2U9RmFsc2UpCiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGVuZ2luZS5yb3dzKClbMF1bInN0YXR1cyJdLCBzdGF0dXMpCiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydEZhbHNlKGVuZ2luZS5yb3dzKClbMF1bImNvcnJlY3QiXSkKICAgICAgICAgICAgICAgIGVuZ2luZS5ydW4odmVyYm9zZT1GYWxzZSkKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKGNsaWVudC5jYWxscyksIDEpCiAgICAgICAgICAgICAgICBlbmdpbmUubWF4X2FwaV9jYWxscyA9IDIKICAgICAgICAgICAgICAgIGVuZ2luZS5ydW4odmVyYm9zZT1GYWxzZSkKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKGNsaWVudC5jYWxscyksIDIpCiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKGNsaWVudC5jYWxsc1swXSwgY2xpZW50LmNhbGxzWzFdKQogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnROb3RJbigiU0VDUkVUX1ZBTFVFX1NIT1VMRF9OT1RfQkVfTE9HR0VEIiwgKFBhdGgoZGlyZWN0b3J5KSAvICJsZWRnZXIuanNvbmwiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgogICAgZGVmIHRlc3RfdW5rbm93bl9zdGFydF9pc19uZXZlcl9yZXNlbnRfYW5kX2NvbnN1bWVzX2J1ZGdldChzZWxmKToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeShkaXI9VEVNUF9ST09UKSBhcyBkaXJlY3Rvcnk6CiAgICAgICAgICAgIGNsaWVudCA9IE1vY2tDbGllbnQoc2VsZi5qb2JzKQogICAgICAgICAgICBlbmdpbmUgPSBzZWxmLm1ha2VfZW5naW5lKGRpcmVjdG9yeSwgY2xpZW50LCBidWRnZXQ9MSkKICAgICAgICAgICAgam9iID0gZW5naW5lLmpvYnNbMF0KICAgICAgICAgICAgZW5naW5lLmFwcGVuZCh7ImV2ZW50IjogImNhbGxfc3RhcnQiLCAiYXR0ZW1wdF9pZCI6ICJ1bmtub3duLXRlc3QiLCAiam9iX2lkIjogam9iWyJqb2JfaWQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgInJlcXVlc3Rfc2hhMjU2Ijogc3IuZGlnZXN0KGpvYlsicmVxdWVzdCJdKSwgInJlcXVlc3QiOiBqb2JbInJlcXVlc3QiXX0pCiAgICAgICAgICAgIHN1bW1hcnkgPSBlbmdpbmUucnVuKHZlcmJvc2U9RmFsc2UpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoY2xpZW50LmNhbGxzLCBbXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlbmdpbmUucm93cygpWzBdWyJzdGF0dXMiXSwgInVua25vd24iKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcnlbInRvdGFscyJdWyJhcGlfYXR0ZW1wdHMiXSwgMSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW1tYXJ5WyJ0b3RhbHMiXVsidW5yZXNvbHZlZF9jYWxscyJdLCAxKQogICAgICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHN1bW1hcnlbImFsbF9maW5pc2hlZCJdKQoKICAgIGRlZiB0ZXN0X2NvbnRyb2xsZXJfcnVsZXNfYW5kX2luY29tcGxldGVfb3V0cHV0cyhzZWxmKToKICAgICAgICBqb2IgPSBuZXh0KGogZm9yIGogaW4gc2VsZi5qb2JzIGlmIGpbInN0dWR5Il0gPT0gIkIiKQogICAgICAgIGNoYW5nZWQgPSBuZXh0KGxldHRlciBmb3IgbGV0dGVyIGluIGpvYlsicGF5bG9hZCJdWyJvcHRpb25zIl0gaWYgbGV0dGVyIG5vdCBpbiBqb2JbImNhbmRpZGF0ZSJdWyJhbnN3ZXIiXSkKICAgICAgICBvYmogPSB7ImFuc3dlciI6IFtjaGFuZ2VkXSwgImJyaWVmX3JlYXNvbiI6ICJ4IiwgImFjdGlvbiI6ICJmaW5pc2giLCAiZGVjaXNpb25fcmVhc29uIjogIngifQogICAgICAgIGZpbmlzaCA9IHsic3RhdHVzIjogIm9rIiwgInJlc3BvbnNlIjogeyJjb250ZW50IjoganNvbi5kdW1wcyhvYmopLCAiZmluaXNoX3JlYXNvbiI6ICJzdG9wIn19CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzci5fZXZhbHVhdGUoam9iLCBmaW5pc2gpWyJzdGF0dXMiXSwgInBhcnNlX2Vycm9yIikKICAgICAgICBvYmpbImFuc3dlciJdID0gam9iWyJjYW5kaWRhdGUiXVsiYW5zd2VyIl0KICAgICAgICBmaW5pc2hbInJlc3BvbnNlIl1bImNvbnRlbnQiXSA9IGpzb24uZHVtcHMob2JqKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3IuX2V2YWx1YXRlKGpvYiwgZmluaXNoKVsic3RhdHVzIl0sICJjb21wbGV0ZWQiKQogICAgICAgIGZpbmlzaFsicmVzcG9uc2UiXVsiZmluaXNoX3JlYXNvbiJdID0gImxlbmd0aCIKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNyLl9ldmFsdWF0ZShqb2IsIGZpbmlzaClbInN0YXR1cyJdLCAicGFyc2VfZXJyb3IiKQoKICAgIGRlZiB0ZXN0X3R3b19zaWRlZF9leGFjdF9tY25lbWFyX2FuZF9ob2xtKHNlbGYpOgogICAgICAgIHJvd3MgPSBbXQogICAgICAgIGZvciBpbmRleCBpbiByYW5nZSg4KToKICAgICAgICAgICAgcm93cyArPSBbeyJjb25kaXRpb24iOiAidjIiLCAidW5pdF9pZCI6IHN0cihpbmRleCksICJzdGF0dXMiOiAiY29tcGxldGVkIiwgImNvcnJlY3QiOiBUcnVlfSwKICAgICAgICAgICAgICAgICAgICAgeyJjb25kaXRpb24iOiAidjEiLCAidW5pdF9pZCI6IHN0cihpbmRleCksICJzdGF0dXMiOiAiY29tcGxldGVkIiwgImNvcnJlY3QiOiBGYWxzZX1dCiAgICAgICAgcmVzdWx0ID0gc3IuU3VwcGxlbWVudGFsUnVuLnBhaXJlZChyb3dzLCAidjIiLCAidjEiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJtY25lbWFyX2V4YWN0X3R3b19zaWRlZF9wIl0sIDIgLyAyNTYpCiAgICAgICAgYWRqdXN0ZWQgPSBzci5TdXBwbGVtZW50YWxSdW4uaG9sbV90d28oeyJhIjogcmVzdWx0LCAiYiI6IHsiYWxsX2ZpbmlzaGVkIjogVHJ1ZSwgIm1jbmVtYXJfZXhhY3RfdHdvX3NpZGVkX3AiOiAwLjAzfX0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChhZGp1c3RlZFsiYSJdWyJob2xtX2FkanVzdGVkX3BfdHdvX2NvbXBhcmlzb25fZmFtaWx5Il0sIDQgLyAyNTYpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChhZGp1c3RlZFsiYiJdWyJob2xtX2FkanVzdGVkX3BfdHdvX2NvbXBhcmlzb25fZmFtaWx5Il0sIDAuMDMpCgogICAgZGVmIHRlc3RfZ3VhcmRyYWlsc19wcm90b2NvbF9jbGllbnRfYW5kX3RydW5jYXRlZF9sZWRnZXIoc2VsZik6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoZGlyPVRFTVBfUk9PVCkgYXMgZGlyZWN0b3J5OgogICAgICAgICAgICBwYXRoID0gUGF0aChkaXJlY3RvcnkpCiAgICAgICAgICAgIGNsaWVudCA9IE1vY2tDbGllbnQoc2VsZi5qb2JzKQogICAgICAgICAgICBjbGllbnQubWF4X3JldHJpZXMgPSAyCiAgICAgICAgICAgIGVuZ2luZSA9IHNlbGYubWFrZV9lbmdpbmUocGF0aCAvICJvdXQiLCBjbGllbnQpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBlbmdpbmUucnVuKHZlcmJvc2U9RmFsc2UpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoY2xpZW50LmNhbGxzLCBbXSkKICAgICAgICAgICAgYWx0ZXJlZCA9IGpzb24ubG9hZHMoUFJPVE9DT0wucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICBhbHRlcmVkWyJwcm90b2NvbCJdWyJtYXhfYXBpX2NhbGxzIl0gPSA5OTkKICAgICAgICAgICAgKHBhdGggLyAiYmFkLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoYWx0ZXJlZCksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBzci5sb2FkX3Byb3RvY29sKHBhdGggLyAiYmFkLmpzb24iLCBGT1JNQUwpCiAgICAgICAgICAgIChwYXRoIC8gInRydW5jYXRlZC5qc29ubCIpLndyaXRlX2J5dGVzKGIneyJldmVudCI6ImNhbGxfc3RhcnQifScpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBzci5yZWFkX2pzb25sKHBhdGggLyAidHJ1bmNhdGVkLmpzb25sIikKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhGaWxlRXhpc3RzRXJyb3IpOgogICAgICAgICAgICAgICAgc3IucHJlcGFyZV9wcm90b2NvbChGT1JNQUwsIFJPT1QgLyAic3VwcGxlbWVudGFsX2l0ZW1zLmpzb24iLCBQUk9UT0NPTCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'sha256': 'c29ab059a292f7cc32e8508db6f0d5893d6674f4890d3ddca917ca306e80ffc3'}, 'task1_smoke/1.exam.json': {'base64': '...', 'sha256': 'ac196322dd8a9e4b37ab371087d7feb640b54c80af80c42ea807934b84211c57'}, 'data_quality_review.json': {'base64': '...', 'sha256': 'e2a6e886b215967f8c98b942a90b9a096cd196783741e1f1d60073c606ec3745'}}
for relative_name, record in EMBEDDED_FILES.items():
    raw = base64.b64decode(record['base64'])
    if hashlib.sha256(raw).hexdigest() != record['sha256']:
        raise RuntimeError('内嵌文件校验失败：' + relative_name)
    target = PROJECT_DIR / relative_name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_bytes() != raw:
        raise RuntimeError('已有不同版本文件，请使用新项目目录：' + str(target))
    target.write_bytes(raw)
RESTORED_RESULTS = {'base64_zip': '...', 'sha256': 'f9f4e0f74de3dd467f3e80f20cd147073da8eb61b28ed565e9a86c4f3ce7dcd6', 'files_sha256': {'recorded_environment.json': 'd951a2f34dc2335eeaf23db7fdd02a14b6db800ee0fcc6b0b51f066c3044146c', 'task1_assignment/analysis/accuracy_comparison.pdf': '5ae3e59ec90461240b5e666a2f367a609d1b5486a1a320cdc19388aa3f9976a5', 'task1_assignment/analysis/accuracy_comparison.png': '3b35635061612b4c2e4d87653ab82e1504f5dfc7123757660fbe9c8553d163db', 'task1_assignment/analysis/agent_changes.csv': '881b111322f8de31674ad14860590b5da0ac5051b56a9669ef682ec0e1539788', 'task1_assignment/analysis/analysis.json': '7612d58a50eb9c96a5dcc44ceb5aa4068117d2d180a1c9dadf624fbe8e1f749b', 'task1_assignment/analysis/case_candidates.json': '2fd62a6f226f605a411e6ac4763803beaa151f60c0b191cff3205dc21cd91523', 'task1_assignment/analysis/efficiency_comparison.pdf': 'd42766ccaf76dfc926fb08d3d8a64cb9657de60491d15686d90d64e6812094e3', 'task1_assignment/analysis/efficiency_comparison.png': '3c47cc7a7d95a9edf9d980ef6e49e6932ac3315d824fc716149dd4dcf2f4067a', 'task1_assignment/analysis/main_table.csv': '7be4ddd03d07ca10df817d20ce75b6abd3a9eed668dfb17501847711c2ed7c1b', 'task1_assignment/analysis/question_type_table.csv': '87adf0c99a4352a41625610010b04e322b3430f109aaf8bddd55f130879b2f7f', 'task1_assignment/development/data_audit.json': '4ca6a99f5f38523396d4db4ae581f2a1d012bcd88e55d8b62ae6270b736427f1', 'task1_assignment/development/ledger.jsonl': 'ecd49f090a941976ffb74db80ae06358cf86969e62a7c85f4b435ce988cf9f78', 'task1_assignment/development/manifest.json': '53a1375c07c5c1665d1b02b2b01a7e72c09d7cb695ca931c2bc8f7d72428e077', 'task1_assignment/development/results.jsonl': '75472b8e5feb3a85eaa2d27955fb079ddc2dfac90ce67d241df379653de05d18', 'task1_assignment/development/source_data.json': '87afcce83e49ce549fc12b88bfdcb6f5fe15fae93ef9410de7cce75324a3c22f', 'task1_assignment/development/summary.csv': '28ec2160e46bdfd0fe6ad28c7ab601cf257352113cd7d1ce64460a9f23abf53c', 'task1_assignment/development/summary.json': '78e288ce39f64a75b5dd99af617c3eb5eac335dc63a01fd97a379bcf5bc5fdd0', 'task1_assignment/formal/data_audit.json': 'd1f0fec7321a3dbf04872d17ce371ca3a1ed4c7d593e8c94e20f461def21b8a5', 'task1_assignment/formal/ledger.jsonl': '7e7dfb205ee3746d8708f7edf013886ef93ee0b8f3e8d0ac357d840a7442720d', 'task1_assignment/formal/manifest.json': '62f4019ca8a2586586983ea762798edeb88195a717db696fdaa7ee8974d4ac32', 'task1_assignment/formal/results.jsonl': '88e4c30b02f78696922ddb1fb4b53e7cbf119cc63976e0b29ae7d86dfe05f10f', 'task1_assignment/formal/source_data.json': 'e685d820983df7f72bb74fc63a2c3da9d156c0668eacd31efc98e6d2e0ec9702', 'task1_assignment/formal/summary.csv': '649d4d603f2ffdcde87f56a1e3ca428c32e52fd7d0f26c8be2c4c3c6fd54915e', 'task1_assignment/formal/summary.json': 'd92159d98e72455e9e4a843cde4a7c0ed9468379b56db5b90ef150dd02070413', 'task1_assignment/supplemental/job_manifest.json': '2bc6aa714186d5f76c042817fddab44bb4c28a4278950e4b153995294c0a058e', 'task1_assignment/supplemental/ledger.jsonl': 'abf1db89228c635ec0307dd65877418cacbeea69a0fe01f9a5f766dc9955128f', 'task1_assignment/supplemental/protocol.json': 'f14d3e403ec7e38b43596c8a6ea14ce36d933f5c1addd12f7abf413a0beee802', 'task1_assignment/supplemental/results.jsonl': '126a3a2e87aa5eecdc00e66ddd63154e3e5f0e661fc4db45ff56309ffd978f97', 'task1_assignment/supplemental/summary.json': '6ed8e4b0dd8ad0d94b98180a4a79537a0c665d055f86007b555a99bb52bac48b', 'task1_assignment/supplemental_analysis/analysis.json': '7d968536d3af50b5df292a10808bc1d14cfd0839155bdff53a411530ad419a9c', 'task1_assignment/supplemental_analysis/metrics.csv': 'be246004f468c11e0192a014411eab45613ac8c9c10b79b4b05062a03f75f20d', 'task1_assignment/supplemental_analysis/paired_comparisons.json': '644c4a609cc3ffe2b01118f075eb4179bc5ab921e427f7985f0987c7d5ae9e05'}}
ALLOWED_RESTORE_PATHS = set(['task1_assignment/development/manifest.json', 'task1_assignment/development/source_data.json', 'task1_assignment/development/data_audit.json', 'task1_assignment/development/ledger.jsonl', 'task1_assignment/development/results.jsonl', 'task1_assignment/development/summary.json', 'task1_assignment/development/summary.csv', 'task1_assignment/formal/manifest.json', 'task1_assignment/formal/source_data.json', 'task1_assignment/formal/data_audit.json', 'task1_assignment/formal/ledger.jsonl', 'task1_assignment/formal/results.jsonl', 'task1_assignment/formal/summary.json', 'task1_assignment/formal/summary.csv', 'task1_assignment/analysis/analysis.json', 'task1_assignment/analysis/case_candidates.json', 'task1_assignment/analysis/main_table.csv', 'task1_assignment/analysis/question_type_table.csv', 'task1_assignment/analysis/agent_changes.csv', 'task1_assignment/analysis/accuracy_comparison.png', 'task1_assignment/analysis/accuracy_comparison.pdf', 'task1_assignment/analysis/efficiency_comparison.png', 'task1_assignment/analysis/efficiency_comparison.pdf', 'task1_assignment/supplemental/protocol.json', 'task1_assignment/supplemental/job_manifest.json', 'task1_assignment/supplemental/ledger.jsonl', 'task1_assignment/supplemental/results.jsonl', 'task1_assignment/supplemental/summary.json', 'task1_assignment/supplemental_analysis/analysis.json', 'task1_assignment/supplemental_analysis/metrics.csv', 'task1_assignment/supplemental_analysis/paired_comparisons.json', 'recorded_environment.json'])
compressed = base64.b64decode(RESTORED_RESULTS['base64_zip'])
if hashlib.sha256(compressed).hexdigest() != RESTORED_RESULTS['sha256']:
    raise RuntimeError('内嵌日志压缩包校验失败。')
expected_files = RESTORED_RESULTS['files_sha256']
if not set(expected_files) <= ALLOWED_RESTORE_PATHS:
    raise RuntimeError('日志包含白名单之外的路径。')
with zipfile.ZipFile(io.BytesIO(compressed)) as bundle:
    names = bundle.namelist()
    if len(names) != len(set(names)) or set(names) != set(expected_files):
        raise RuntimeError('日志压缩包文件名单不一致。')
    for relative_name in names:
        target = (PROJECT_DIR / relative_name).resolve()
        if not target.is_relative_to(PROJECT_DIR.resolve()):
            raise RuntimeError('拒绝项目目录之外的恢复路径。')
        raw = bundle.read(relative_name)
        if hashlib.sha256(raw).hexdigest() != expected_files[relative_name]:
            raise RuntimeError('日志文件校验失败：' + relative_name)
        if not RESTORE_EXISTING_RESULTS:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != raw:
            if relative_name.startswith(('task1_assignment/analysis/', 'task1_assignment/supplemental_analysis/')):
                continue
            raise RuntimeError('已有不同日志，拒绝覆盖；请改用新PROJECT_DIR：' + relative_name)
        target.write_bytes(raw)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
environment = {'python': sys.version, 'openai': version('openai'), 'matplotlib': version('matplotlib'), 'embedded_sha256': {name: record['sha256'] for name, record in EMBEDDED_FILES.items()}}
Path('environment.json').write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8')
print('冻结源码与数据已校验。项目目录：', PROJECT_DIR)
print('已附真实日志/分析文件数：', len(expected_files), '；本次启用恢复：', RESTORE_EXISTING_RESULTS)
print('本格没有调用模型API。')


## 3. 读取数据并建立实验入口

导入驱动只读取数据，不调用模型。完整数据100题：3条选项全空，主评测97条；另外7条预注册的图示/上下文风险记录用于90题敏感性分析。两套成绩使用同一批预测，不额外选择有利样本。

In [ ]:
from cloud_driver import DATA, DEV_DATA, ROOT, run_development, run_formal
from collections import Counter

print("正式原始记录：", len(DATA))
print("题型：", dict(Counter(q["question_type"] for q in DATA)))
print("独立人工构造开发题：", len(DEV_DATA))
print("请勿把正式题的answer或analysis加入提示或Agent审查。")

## 4. 是否允许开始/恢复付费实验，以及隐藏输入Key

阅读并离线复算已附结果时两个开关都保持`False`。主实验由`RUN_LIVE_EXPERIMENTS`控制，补充研究由`RUN_SUPPLEMENTAL_LIVE`独立控制；任一开启才显示隐藏Key输入。已完成项不会重新调用。主实验独立重跑才需要新的`PROJECT_DIR`并设置`RESTORE_EXISTING_RESULTS=False`；补充重复运行必须保留协议冻结的原formal状态并设置新输出目录，不能混用新跑的主实验。密钥只用于内存中的客户端，不自动重试。

参考：[DeepSeek官方调用说明](https://api-docs.deepseek.com/) · [思考模式](https://api-docs.deepseek.com/guides/thinking_mode/) · [JSON模式](https://api-docs.deepseek.com/guides/json_mode/)

In [ ]:
from getpass import getpass
from openai import OpenAI

RUN_LIVE_EXPERIMENTS = False
RUN_SUPPLEMENTAL_LIVE = False
_assignment2_client = None
if RUN_LIVE_EXPERIMENTS or RUN_SUPPLEMENTAL_LIVE:
    _key_input = getpass("输入DeepSeek官方API Key（隐藏输入）：").strip()
    if not _key_input:
        raise ValueError("没有输入Key；没有启动实验。")
    _assignment2_client = OpenAI(
        api_key=_key_input,
        base_url="https://api.deepseek.com",
        timeout=60.0,
        max_retries=0,
    )
    del _key_input
    print("客户端已配置；接下来先运行独立开发检查。")
else:
    print("只读模式：不会请求API，也不会索取Key。")

## 5. 独立开发检查

三道开发题不来自正式测试题，也不同于few-shot示例。只验证负向问法、单多选及输出格式，不用于报告真实考试准确率。最多24次调用，失败会阻止正式实验启动。

In [ ]:
if RUN_LIVE_EXPERIMENTS:
    dev_engine = run_development(_assignment2_client)
else:
    print("已跳过开发API调用。")

## 6. 正式四组实验

四组为Zero-shot、Few-shot、CoT、反馈驱动Agent。Agent使用与CoT完全相同的初答提示，再由独立审查产生反馈，控制器根据反馈选择提交或修订，最多两轮复核。

统一模型`deepseek-flash`，关闭内置思考模式，温度0，单次输出上限768 token。四组正常最多776次API调用；正式累计上限800留给明确需要的恢复，程序不会隐性重试。每次调用和每题完成都会保存日志、打印进度。

重新运行同一配置会跳过已有完成结果；失败/未确认请求不会自动重发。不要根据测试答案改提示后复用旧实验目录。

In [ ]:
if RUN_LIVE_EXPERIMENTS:
    formal_engine = run_formal(_assignment2_client)
else:
    print("已跳过正式API调用。只读摘要不会被当作本notebook刚执行的结果。")

## 6b. 主实验之后的补充研究（独立开关）

16道人工封闭世界检查的原CoT/v2比较共32次调用；95个固定合法初答的feedback/null控制器比较共190次调用。两组B请求仅反馈字段不同，均没有后续复核轮次。原第14、67条初答无效，在97题口径下两组共同计错。两项新增主比较为A全部16题与B条件95题，单独采用Holm2校正；这是事后补充证据，不替代原主实验。

默认离线。允许补充调用时，`SUPPLEMENTAL_FORMAL_DIR`必须保持原冻结formal日志；同目录中任何已尝试项（包括失败）都不会再次调用。若要做一次新的完整重复运行，保留原formal目录，仅把`SUPPLEMENTAL_OUTPUT_DIR`改为新的空目录，并开启补充开关；不要按答错情况挑题重复。

In [ ]:
from supplemental_runner import run_supplemental

SUPPLEMENTAL_FORMAL_DIR = ROOT / "formal"
SUPPLEMENTAL_OUTPUT_DIR = ROOT / "supplemental"
if RUN_SUPPLEMENTAL_LIVE:
    supplemental_engine = run_supplemental(
        _assignment2_client,
        protocol_path=PROJECT_DIR / "supplemental_protocol.json",
        formal_dir=SUPPLEMENTAL_FORMAL_DIR,
        output_dir=SUPPLEMENTAL_OUTPUT_DIR,
        max_api_calls=240,
    )
else:
    print("已跳过补充研究API调用。")

## 7. 从真实日志离线生成统计与图表

此格不调用API。验证原始数据、指纹、ledger、逐题结果一致后，生成97/90题成绩、95% Wilson区间、配对McNemar检验及Holm校正、API时间/token统计、Agent初答到最终答案变化和真实案例候选。正式记录未完成时停止，不生成假成绩。

In [ ]:
from analyze_results import load_run, write_outputs

formal_dir = ROOT / "formal"
if (formal_dir / "summary.json").exists():
    checked_run = load_run(formal_dir, ROOT / "1.exam.json")
    analysis = write_outputs(checked_run, ROOT / "analysis")
    print("离线分析已保存到：", ROOT / "analysis")
    for row in analysis["main_results"]:
        print(row["subset"], row["method"], f"{row['correct']}/{row['n']}", f"{row['accuracy']:.2%}")
else:
    print("当前运行时没有正式日志。可阅读上面的真实日志摘要，或重新运行实验后再分析。")

from analyze_supplemental import load_supplemental, write_outputs as write_supplemental_outputs

if (SUPPLEMENTAL_OUTPUT_DIR / "summary.json").exists():
    checked_supplemental = load_supplemental(
        SUPPLEMENTAL_OUTPUT_DIR, SUPPLEMENTAL_FORMAL_DIR, PROJECT_DIR / "supplemental_protocol.json")
    supplemental_analysis = write_supplemental_outputs(checked_supplemental, ROOT / "supplemental_analysis")
    for study, conditions in (("A", ("cot_v1", "cot_v2")), ("B", ("feedback", "null_feedback"))):
        subset = "all_16" if study == "A" else "conditional_95"
        for condition in conditions:
            row = supplemental_analysis["study_" + study][condition][subset]
            print("补充", study, condition, subset, f"{row['correct']}/{row['n']}", row["accuracy"])
    print("新增两项主比较（双侧McNemar、Holm2）：", supplemental_analysis["supplementary_primary_comparisons"])
else:
    print("当前目录没有补充日志；不填写补充成绩，也不调用API。")

## 8. 导出源码、数据、日志和分析

此格只打包明确列出的项目文件，不打包客户端、Key或整个运行时。主实验或补充日志尚未完整时，文件名带`partial`，两项状态分开记录。Colab临时运行时可能回收，请下载保存结果。

In [ ]:
import zipfile
from google.colab import files

RESULT_NAMES = ('manifest.json', 'source_data.json', 'data_audit.json', 'ledger.jsonl', 'results.jsonl', 'summary.json', 'summary.csv')
ANALYSIS_NAMES = ('analysis.json', 'case_candidates.json', 'main_table.csv', 'question_type_table.csv', 'agent_changes.csv', 'accuracy_comparison.png', 'accuracy_comparison.pdf', 'efficiency_comparison.png', 'efficiency_comparison.pdf')
SUPPLEMENTAL_NAMES = ('protocol.json', 'job_manifest.json', 'ledger.jsonl', 'results.jsonl', 'summary.json')
SUPPLEMENTAL_ANALYSIS_NAMES = ('analysis.json', 'metrics.csv', 'paired_comparisons.json')
summary_path = ROOT / "formal" / "summary.json"
finished = False
if summary_path.exists():
    status = json.loads(summary_path.read_text(encoding="utf-8"))
    finished = all(v.get("all_valid_inputs_finished") for v in status.get("methods", {}).values()) and len(status.get("methods", {})) == 4
supplemental_finished = False
if (SUPPLEMENTAL_OUTPUT_DIR / "summary.json").exists():
    supplemental_finished = json.loads((SUPPLEMENTAL_OUTPUT_DIR / "summary.json").read_text(encoding="utf-8")).get("all_finished") is True
archive = PROJECT_DIR / ("Task1_results_complete.zip" if finished and supplemental_finished else "Task1_results_partial.zip")
export_entries = {}
for name in ('runner.py', 'cloud_driver.py', 'analyze_results.py', 'supplemental_runner.py', 'supplemental_protocol.json', 'supplemental_items.json', 'analyze_supplemental.py', 'test_supplemental_runner.py', 'environment.json', 'recorded_environment.json', 'data_quality_review.json'):
    path = PROJECT_DIR / name
    if path.is_file():
        export_entries["assignment2_task1/" + name] = path
export_entries["data/1.exam.json"] = PROJECT_DIR / "task1_smoke/1.exam.json"
for phase in ("development", "formal"):
    for name in RESULT_NAMES:
        path = ROOT / phase / name
        if path.is_file():
            export_entries[f"results/{phase}/{name}"] = path
for name in ANALYSIS_NAMES:
    path = ROOT / "analysis" / name
    if path.is_file():
        export_entries["analysis/" + name] = path
for name in SUPPLEMENTAL_NAMES:
    path = SUPPLEMENTAL_OUTPUT_DIR / name
    if path.is_file():
        export_entries["results/supplemental/" + name] = path
for name in SUPPLEMENTAL_ANALYSIS_NAMES:
    path = ROOT / "supplemental_analysis" / name
    if path.is_file():
        export_entries["supplemental_analysis/" + name] = path
supplemental_formal_archive = "results/formal"
if SUPPLEMENTAL_FORMAL_DIR.resolve() != (ROOT / "formal").resolve():
    supplemental_formal_archive = "results/supplemental_original_formal"
    for name in ("manifest.json", "source_data.json", "ledger.jsonl", "results.jsonl"):
        path = SUPPLEMENTAL_FORMAL_DIR / name
        if path.is_file():
            export_entries[supplemental_formal_archive + "/" + name] = path
runtime_manifest = {
    "formal_run_status": "complete" if finished else "partial",
    "supplemental_run_status": "complete" if supplemental_finished else "partial",
    "supplemental_formal_archive": supplemental_formal_archive,
    "contains_notebook": False,
    "included_tests": [name for name in export_entries if "/test_" in name],
    "files_sha256": {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in export_entries.items()},
}
runtime_readme = (
    "Task 1 运行时结果导出包\n\n"
    + "正式实验状态：" + runtime_manifest["formal_run_status"] + "\n"
    + "补充研究状态：" + runtime_manifest["supplemental_run_status"] + "\n"
    + "本ZIP只包含下面列出的源码、数据、日志、测试和已生成分析；不包含notebook。\n"
    + "API Key和客户端对象没有打包。environment.json为当前复算环境；recorded_environment.json若存在则来自原始运行记录。\n\n"
    + "离线复算（无需API Key；仅完整正式日志可生成最终分析）：\n"
    + "python -m pip install --index-url https://pypi.org/simple matplotlib\n"
    + "python assignment2_task1/analyze_results.py --run-dir results/formal --data data/1.exam.json --output-dir analysis_reproduced\n\n"
    + "python assignment2_task1/analyze_supplemental.py --run-dir results/supplemental --formal-dir " + supplemental_formal_archive + " --output-dir supplemental_analysis_reproduced\n\n"
    + "重跑模型需要另外打开完整交付notebook；同配置已完成项会跳过。\n"
    + "实际文件清单（逐文件SHA见reproduction_manifest.json）：\n"
    + "\n".join("- " + name for name in sorted(export_entries))
    + "\n- README.txt\n- reproduction_manifest.json\n"
)
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as package:
    package.writestr("README.txt", runtime_readme)
    package.writestr("reproduction_manifest.json", json.dumps(runtime_manifest, ensure_ascii=False, indent=2))
    for name, path in export_entries.items():
        package.write(path, name)
print("已打包：", archive.name)
print("正式实验完整：", finished)
print("补充研究完整：", supplemental_finished)
files.download(str(archive))

## 报告如何使用这些输出

- `manifest.json`给出实际模型、参数、提示、示例、数据和代码指纹。
- `main_table.csv`和两张比较图用于结果部分；准确率与成本一起报告。
- `agent_changes.csv`区分改对、改错和执行失败；不能只挑成功案例。
- `case_candidates.json`只选实际出现的案例，不存在的改进类型保持空。
- 正式数据缺项、few-shot示例知识难度、样本规模和模型随机性均需讨论。
- `API/parse`失败留在有效输入分母中计错；费用为声明费率下的估计，不是账单。
- `supplemental_analysis`提供事后追加研究的两项对比及Holm2；不能与主实验原比较族混称预先设计，也不能将单步消融描述成完整Agent消融。

本notebook不自动提交Blackboard；LaTeX报告是单独交付物。